In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018Q4.csv.gz
/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz
/kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv
/kaggle/input/datasets/wordsforthewise/lending-club/rejected_2007_to_2018q4.csv/rejected_2007_to_2018Q4.csv


In [2]:
#!pip install --upgrade torch torchvision torchaudio

In [3]:
# ============================================================
# BLOCK 1: IMPORTS, PATHS, CONFIGURATION
# ============================================================

import os
import re
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
#from datasets import Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score,
    precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    matthews_corrcoef, brier_score_loss, log_loss,
    confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.calibration import calibration_curve
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import spearmanr
from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import get_linear_schedule_with_warmup

sns.set_theme(style="whitegrid")

# --- Reproducibility ---
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# --- Paths (Kaggle-compatible) ---
BASE_DIR = "/kaggle/input/datasets/wordsforthewise/lending-club"
DATA_ROOT = BASE_DIR
WORKING_DIR = "/kaggle/working"

LENDING_CLUB_FILE = os.environ.get(
    "INPUT_PATH_MAIN",
    os.path.join(DATA_ROOT, "accepted_2007_to_2018Q4.csv.gz")
)
PROSPER_FILE = os.environ.get(
    "INPUT_PATH_SECONDARY",
    os.path.join(DATA_ROOT, "prosper_loan_dataset.csv")
)

PROCESSED = os.path.join(WORKING_DIR, "bankshield_processed_rich.csv")
TRAIN_PATH = os.path.join(WORKING_DIR, "train_rich.csv")
VAL_PATH = os.path.join(WORKING_DIR, "val_rich.csv")
TEST_PATH = os.path.join(WORKING_DIR, "test_rich.csv")
SCALER_PATH = os.path.join(WORKING_DIR, "bankshield_numeric_scaler.pkl")
ADV_PATH = os.path.join(WORKING_DIR, "adversarial_test.csv")
BASELINE_DIR = os.path.join(WORKING_DIR, "baseline_model")
REGULATORY_DIR = os.path.join(WORKING_DIR, "regulatory_model")

os.makedirs(WORKING_DIR, exist_ok=True)
os.makedirs(BASELINE_DIR, exist_ok=True)
os.makedirs(REGULATORY_DIR, exist_ok=True)

# --- Shared config ---
MODEL_NAME = "ProsusAI/finbert"
TRAIN_SAMPLE = 50_000          # reduced from 200k for faster iteration
NUM_EPOCHS = 5              # change to 5 for full training
MAX_SEQ_LEN = 96
BATCH_SIZE = 64
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- HPO defaults (will be overridden by HPO sweep in Block 8) ---
LAMBDA_REG = 0.5               # conservative starting point
TAU_TRAIN = 0.5                # penalty threshold during training
TAU_DECISION = 0.35             # operating threshold for predictions

# --- Feature columns ---
NUM_COLS = [
    "annual_inc", "dti", "loan_amnt",
    "loan_to_income", "income_log", "loan_log",
    "monthly_income", "monthly_debt_est", "monthly_payment_est"
]

FLAG_COLS = [
    "flag_dti_43", "flag_lti_high", "flag_pti_high",
    "flag_predatory", "flag_dual_stress", "flag_debt_burden",
    "flag_large_loan_dti"
]

print("Block 1 complete.")

Block 1 complete.


### Build processed dataset from raw CSV

In [4]:
# ============================================================
# BLOCK 2: DATA LOADING & PREPROCESSING
# ============================================================
 
# --- Load raw Lending Club data ---
print(f"Loading Lending Club data from: {LENDING_CLUB_FILE}")
raw_df = pd.read_csv(LENDING_CLUB_FILE, low_memory=False)
print(f"Raw data shape: {raw_df.shape}")
 
# --- Initial filtering ---
# Keep only fully paid or charged off loans (exclude current/in progress)
raw_df = raw_df[raw_df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
print(f"After status filter: {raw_df.shape}")
 
# --- Create binary label ---
raw_df['label'] = (raw_df['loan_status'] == 'Charged Off').astype(int)
print(f"Label distribution:\n{raw_df['label'].value_counts()}")
 
# --- Core numeric imputation ---
raw_df['annual_inc'] = raw_df['annual_inc'].fillna(raw_df['annual_inc'].median())
raw_df['dti']        = raw_df['dti'].fillna(raw_df['dti'].median())
raw_df['loan_amnt']  = raw_df['loan_amnt'].fillna(raw_df['loan_amnt'].median())
 
# Guard against zero/negative income before any division
raw_df = raw_df[raw_df['annual_inc'] > 0].copy()
 
# --- Derived numeric features ---
raw_df['loan_to_income']      = (raw_df['loan_amnt'] / raw_df['annual_inc']).clip(upper=50)
raw_df['income_log']          = np.log1p(raw_df['annual_inc'].clip(lower=0))
raw_df['loan_log']            = np.log1p(raw_df['loan_amnt'].clip(lower=0))
raw_df['monthly_income']      = raw_df['annual_inc'] / 12.0
raw_df['monthly_debt_est']    = raw_df['monthly_income'] * (raw_df['dti'] / 100.0)
raw_df['monthly_payment_est'] = raw_df['loan_amnt'] / 36.0  # assume 3-year term
 
# --- Rich text field for FinBERT (keep categorical signals the transformer can reason about) ---
# Clean up each categorical column first
for col in ['purpose', 'emp_title', 'home_ownership', 'verification_status', 'grade', 'term']:
    if col in raw_df.columns:
        raw_df[col] = raw_df[col].fillna('unknown').astype(str).str.strip()
 
# Parse int_rate cleanly (may come as "11.49%" string)
if 'int_rate' in raw_df.columns:
    raw_df['int_rate_num'] = pd.to_numeric(
        raw_df['int_rate'].astype(str).str.replace('%', '', regex=False),
        errors='coerce'
    ).fillna(raw_df['int_rate'].apply(pd.to_numeric, errors='coerce').median())
else:
    raw_df['int_rate_num'] = 0.0
 
raw_df['text'] = (
    "Loan purpose: "          + raw_df['purpose'] + ". "
    "Employment: "            + raw_df['emp_title'] + ". "
    "Home ownership: "        + raw_df['home_ownership'] + ". "
    "Verification status: "   + raw_df['verification_status'] + ". "
    "Loan grade: "            + raw_df['grade'] + ". "
    "Interest rate: "         + raw_df['int_rate_num'].round(2).astype(str) + "%. "
    "Term: "                  + raw_df['term'] + ". "
    "Annual income: $"        + raw_df['annual_inc'].round(0).astype(int).astype(str) + ". "
    "Debt-to-income ratio: "  + raw_df['dti'].round(2).astype(str) + ". "
    "Loan amount: $"          + raw_df['loan_amnt'].round(0).astype(int).astype(str) + "."
)
 
# --- Regulatory flags ---
# FLAG 1: DTI > 43% — CFPB Qualified Mortgage hard cap
raw_df['flag_dti_43'] = (raw_df['dti'] > 43).astype(int)
 
# FLAG 2: Loan-to-income > 0.5 — loan exceeds 50% of annual income
raw_df['flag_lti_high'] = (raw_df['loan_to_income'] > 0.5).astype(int)
 
# FLAG 3: Monthly payment > 35% of monthly income (FHA/VA guideline)
pti = raw_df['monthly_payment_est'] / raw_df['monthly_income'].clip(lower=1.0)
raw_df['flag_pti_high'] = (pti > 0.35).fillna(0).astype(int)
 
# FLAG 4: Predatory lending — low income (<$20k/yr) with large loan (>$10k)
raw_df['flag_predatory'] = (
    (raw_df['annual_inc'] < 20_000) & (raw_df['loan_amnt'] > 10_000)
).astype(int)
 
# FLAG 5: Dual stress — already stretched DTI AND high LTI
raw_df['flag_dual_stress'] = (
    (raw_df['dti'] > 36) & (raw_df['loan_to_income'] > 0.3)
).astype(int)
 
# FLAG 6: Severe existing debt burden (monthly debt > 50% of income before this loan)
raw_df['flag_debt_burden'] = (
    raw_df['monthly_debt_est'] / raw_df['monthly_income'].clip(lower=1.0) > 0.50
).astype(int)
 
# FLAG 7: Large loan (top quartile) on already-stretched borrower
loan_75pct = raw_df['loan_amnt'].quantile(0.75)
raw_df['flag_large_loan_dti'] = (
    (raw_df['loan_amnt'] > loan_75pct) & (raw_df['dti'] > 30)
).astype(int)
 
# --- Composite reg_violation: any flag triggered ---
FLAG_COLS = [
    'flag_dti_43', 'flag_lti_high', 'flag_pti_high',
    'flag_predatory', 'flag_dual_stress', 'flag_debt_burden',
    'flag_large_loan_dti'
]
raw_df['reg_violation'] = (raw_df[FLAG_COLS].sum(axis=1) > 0).astype(int)
 
# --- Weighted violation score (reflects regulatory severity, not just count) ---
VIOLATION_WEIGHTS = {
    'flag_dti_43':        1.0,
    'flag_lti_high':      1.0,
    'flag_pti_high':      1.0,
    'flag_predatory':     2.0,   # higher weight — income/loan mismatch is a serious risk
    'flag_dual_stress':   3.0,   # highest weight — two stress indicators together
    'flag_debt_burden':   2.0,
    'flag_large_loan_dti':1.5,
}
raw_df['violation_score'] = sum(
    raw_df[col] * w for col, w in VIOLATION_WEIGHTS.items()
)
# Normalize to [0, 1] so it sits on the same scale as other numeric features
vs_max = raw_df['violation_score'].max()
if vs_max > 0:
    raw_df['violation_score'] = raw_df['violation_score'] / vs_max
 
# --- Stratified balanced sampling ---
# Stratify by label AND violation tercile so model sees all severity levels
raw_df['_violation_tercile'] = pd.cut(
    raw_df['violation_score'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['low', 'med', 'high'],
    include_lowest=True
)
strat_col = raw_df['label'].astype(str) + '_' + raw_df['_violation_tercile'].astype(str)
 
sample_df, _ = train_test_split(
    raw_df, train_size=TRAIN_SAMPLE, stratify=strat_col, random_state=SEED
)
sample_df = sample_df.drop(columns=['_violation_tercile'])
print(f"Sampled data shape: {sample_df.shape}")
print(f"Label distribution in sample:\n{sample_df['label'].value_counts()}")
print(f"Reg violation rate in sample: {sample_df['reg_violation'].mean():.3f}")
 
# --- Train / val / test split ---
train_df, temp_df = train_test_split(
    sample_df, test_size=0.3, stratify=sample_df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=SEED
)
print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")
 
# --- Sanity check BEFORE scaling ---
print("\nPRE-SCALE CHECK (should show raw magnitudes, e.g. annual_inc ~75k):")
print(train_df[NUM_COLS].describe().loc[['mean', 'std', 'min', 'max']])
 
# --- Fit scaler on train only, then transform all splits ---
scaler = StandardScaler()
scaler.fit(train_df[NUM_COLS])
joblib.dump(scaler, SCALER_PATH)
print(f"Saved scaler to {SCALER_PATH}")
 
# Apply scaling IN MEMORY — do NOT save scaled data to CSV
# (downstream blocks that reload CSVs must call scaler.transform themselves)
train_df[NUM_COLS] = scaler.transform(train_df[NUM_COLS])
val_df[NUM_COLS]   = scaler.transform(val_df[NUM_COLS])
test_df[NUM_COLS]  = scaler.transform(test_df[NUM_COLS])
 
# Sanity check AFTER scaling
print("\nPOST-SCALE CHECK (means should be ~0, stds ~1):")
print(train_df[NUM_COLS].describe().loc[['mean', 'std', 'min', 'max']])
 
# Save UNSCALED splits to disk (scale is re-applied after any reload)
# To do this cleanly, invert the scaling before saving
train_df_raw = train_df.copy()
val_df_raw   = val_df.copy()
test_df_raw  = test_df.copy()
 
train_df_raw[NUM_COLS] = scaler.inverse_transform(train_df[NUM_COLS])
val_df_raw[NUM_COLS]   = scaler.inverse_transform(val_df[NUM_COLS])
test_df_raw[NUM_COLS]  = scaler.inverse_transform(test_df[NUM_COLS])
 
train_df_raw.to_csv(TRAIN_PATH, index=False)
val_df_raw.to_csv(VAL_PATH,     index=False)
test_df_raw.to_csv(TEST_PATH,   index=False)
print(f"Saved UNSCALED splits to {WORKING_DIR}")
 
# train_df / val_df / test_df remain SCALED in memory for Block 3
print("Block 2 complete.")
 
 
# ============================================================
# FEATURE ENGINEERING HELPER (used by Blocks 3, adversarial,
# and any downstream block that reloads from CSV)
# ============================================================
 
def prepare_df(df):
    """
    Applies feature engineering to a raw split loaded from CSV.
    Does NOT scale — call scaler.transform(df[NUM_COLS]) separately.
    """
    df = df.copy()
 
    # --- Type coercion ---
    for c in ['purpose', 'emp_title', 'home_ownership',
              'verification_status', 'grade', 'term']:
        if c in df.columns:
            df[c] = df[c].fillna('unknown').astype(str).str.strip()
 
    for c in ['annual_inc', 'dti', 'loan_amnt', 'label']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
 
    df = df.dropna(subset=['annual_inc', 'dti', 'loan_amnt', 'label']).copy()
    df = df[df['annual_inc'] > 0].copy()
 
    # --- Derived numeric features ---
    df['loan_to_income']      = (df['loan_amnt'] / df['annual_inc']).clip(upper=50)
    df['income_log']          = np.log1p(df['annual_inc'].clip(lower=0))
    df['loan_log']            = np.log1p(df['loan_amnt'].clip(lower=0))
    df['monthly_income']      = df['annual_inc'] / 12.0
    df['monthly_debt_est']    = df['monthly_income'] * (df['dti'] / 100.0)
    df['monthly_payment_est'] = df['loan_amnt'] / 36.0
 
    # --- int_rate cleanup (may be string "11.49%" or numeric) ---
    if 'int_rate' in df.columns:
        df['int_rate_num'] = pd.to_numeric(
            df['int_rate'].astype(str).str.replace('%', '', regex=False),
            errors='coerce'
        ).fillna(0.0)
    else:
        df['int_rate_num'] = 0.0
 
    # --- Rich text for FinBERT ---
    df['text'] = (
        "Loan purpose: "         + df['purpose'].astype(str) + ". "
        "Employment: "           + df['emp_title'].astype(str) + ". "
        "Home ownership: "       + df['home_ownership'].astype(str) + ". "
        "Verification status: "  + df['verification_status'].astype(str) + ". "
        "Loan grade: "           + df['grade'].astype(str) + ". "
        "Interest rate: "        + df['int_rate_num'].round(2).astype(str) + "%. "
        "Term: "                 + df['term'].astype(str) + ". "
        "Annual income: $"       + df['annual_inc'].round(0).astype(int).astype(str) + ". "
        "Debt-to-income ratio: " + df['dti'].round(2).astype(str) + ". "
        "Loan amount: $"         + df['loan_amnt'].round(0).astype(int).astype(str) + "."
    )
 
    # --- Regulatory flags ---
    df['flag_dti_43']     = (df['dti'] > 43).astype(int)
    df['flag_lti_high']   = (df['loan_to_income'] > 0.5).astype(int)
 
    pti = df['monthly_payment_est'] / df['monthly_income'].clip(lower=1.0)
    df['flag_pti_high']   = (pti > 0.35).fillna(0).astype(int)
 
    df['flag_predatory']  = (
        (df['annual_inc'] < 20_000) & (df['loan_amnt'] > 10_000)
    ).astype(int)
 
    df['flag_dual_stress'] = (
        (df['dti'] > 36) & (df['loan_to_income'] > 0.3)
    ).astype(int)
 
    df['flag_debt_burden'] = (
        df['monthly_debt_est'] / df['monthly_income'].clip(lower=1.0) > 0.50
    ).astype(int)
 
    loan_75pct = df['loan_amnt'].quantile(0.75)
    df['flag_large_loan_dti'] = (
        (df['loan_amnt'] > loan_75pct) & (df['dti'] > 30)
    ).astype(int)
 
    flag_cols = [
        'flag_dti_43', 'flag_lti_high', 'flag_pti_high',
        'flag_predatory', 'flag_dual_stress', 'flag_debt_burden',
        'flag_large_loan_dti'
    ]
 
    df['reg_violation'] = (df[flag_cols].sum(axis=1) > 0).astype(int)
 
    VIOLATION_WEIGHTS = {
        'flag_dti_43':        1.0,
        'flag_lti_high':      1.0,
        'flag_pti_high':      1.0,
        'flag_predatory':     2.0,
        'flag_dual_stress':   3.0,
        'flag_debt_burden':   2.0,
        'flag_large_loan_dti':1.5,
    }
    df['violation_score'] = sum(
        df[col] * w for col, w in VIOLATION_WEIGHTS.items()
    )
    vs_max = df['violation_score'].max()
    if vs_max > 0:
        df['violation_score'] = df['violation_score'] / vs_max
 
    return df
 
 

Loading Lending Club data from: /kaggle/input/datasets/wordsforthewise/lending-club/accepted_2007_to_2018Q4.csv.gz
Raw data shape: (2260701, 151)
After status filter: (1345310, 151)
Label distribution:
label
0    1076751
1     268559
Name: count, dtype: int64
Sampled data shape: (50000, 169)
Label distribution in sample:
label
0    40019
1     9981
Name: count, dtype: int64
Reg violation rate in sample: 0.030
Train: (35000, 169), Val: (7500, 169), Test: (7500, 169)

PRE-SCALE CHECK (should show raw magnitudes, e.g. annual_inc ~75k):
        annual_inc         dti     loan_amnt  loan_to_income  income_log  \
mean  7.617655e+04   18.317071  14512.569286         0.22155   11.088968   
std   5.252808e+04   12.884386   8729.715106         0.50232    0.539668   
min   1.000000e+00    0.000000    500.000000         0.00087    0.693147   
max   1.750000e+06  999.000000  40000.000000        50.00000   14.375127   

       loan_log  monthly_income  monthly_debt_est  monthly_payment_est  
mean   

### Train / val / test split

In [5]:
# ============================================================
# BLOCK 3: TOKENIZATION, DATASET, AND HYBRID MODEL ARCHITECTURE
# ============================================================
# ============================================================
# BLOCK 3 PREAMBLE: reload splits and re-apply scaling
# (paste this at the TOP of Block 3, replacing its current
#  pd.read_csv lines)
# ============================================================
 
# --- Reload splits from disk (they are stored unscaled) ---
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)
 
print(f"Loaded — Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
 
# --- Re-apply scaling (scaler was fit on train in Block 2) ---
scaler = joblib.load(SCALER_PATH)
train_df[NUM_COLS] = scaler.transform(train_df[NUM_COLS])
val_df[NUM_COLS]   = scaler.transform(val_df[NUM_COLS])
test_df[NUM_COLS]  = scaler.transform(test_df[NUM_COLS])
 
# Quick sanity check
print("\nPOST-RELOAD SCALE CHECK (means ~0, stds ~1):")
print(train_df[NUM_COLS].describe().loc[['mean', 'std', 'min', 'max']])


# --- Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ============================================================
# DATASET CLASS
# ============================================================
class LoanDataset(Dataset):
    """
    Each sample returns:
      - input_ids, attention_mask   (text → FinBERT)
      - numeric_features            (scaled continuous features)
      - flag_features               (binary regulatory flags)
      - violation_score             (continuous [0,1] regulatory severity)
      - label                       (0 = safe, 1 = risky)
    """
    def __init__(self, dataframe, tokenizer, max_length=MAX_SEQ_LEN):
        # Force to pandas regardless of input type
        if not isinstance(dataframe, pd.DataFrame):
            dataframe = dataframe.to_pandas()
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Text encoding
        encoding = self.tokenizer(
            str(row["text"]),
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        # Numeric features (already scaled)
        numeric = torch.tensor(
            [float(row[c]) for c in NUM_COLS], dtype=torch.float32
        )
        
        # Regulatory flag features
        flags = torch.tensor(
            [float(row[c]) for c in FLAG_COLS], dtype=torch.float32
        )

        # Violation score (continuous regulatory severity)
        violation = torch.tensor(float(row["violation_score"]), dtype=torch.float32)
        label = torch.tensor(float(row["label"]), dtype=torch.float32)

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "numeric": numeric,
            "flags": flags,
            "violation_score": violation,
            "label": label,
        }

# --- Build datasets ---
train_dataset = LoanDataset(train_df, tokenizer)
val_dataset = LoanDataset(val_df, tokenizer)
test_dataset = LoanDataset(test_df, tokenizer)

# --- Build dataloaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataloaders ready — Batch size: {BATCH_SIZE}")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")
print(f"  Test batches:  {len(test_loader)}")

# ============================================================
# HYBRID MODEL: FinBERT + Numeric + Regulatory Flags
# ============================================================
class HybridLoanModel(nn.Module):
    """
    Three-stream architecture:
      Stream 1: FinBERT [CLS] → text embedding (768-d)
      Stream 2: Numeric features → small MLP (num_dim → 64)
      Stream 3: Regulatory flags → small MLP (flag_dim → 32)

    Fusion: concatenate all three → MLP head → P(risky)

    Design choices:
      - FinBERT is FROZEN by default (fine-tune only the head)
        to prevent the text encoder from learning to override
        regulatory signals. This is a deliberate architectural
        guardrail: the model cannot "unlearn" compliance by
        adjusting text representations.
      - Dropout after each stream and in the fusion head for
        regularization.
      - Flag stream is kept shallow (1 hidden layer) so the
        model can't bury flag signals under nonlinearity.
    """

    def __init__(
        self,
        num_dim,
        flag_dim,
        freeze_bert=True,
        dropout=0.3,
        numeric_hidden=64,
        flag_hidden=32,
        fusion_hidden=128,
    ):
        super().__init__()

        # --- Stream 1: FinBERT text encoder ---
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        self.bert_hidden = self.bert.config.hidden_size  # 768

        if freeze_bert:
            for name, param in self.bert.named_parameters():
                # Freeze everything except the last 2 transformer layers and pooler
                if not any(f"layer.{i}" in name for i in [10, 11]) and "pooler" not in name:
                    param.requires_grad = False
            print("  FinBERT: PARTIALLY FROZEN (layers 10-11 + pooler trainable)")
        else:
            print("  FinBERT: FULLY TRAINABLE (fine-tuning enabled)")

        self.text_dropout = nn.Dropout(dropout)

        # --- Stream 2: Numeric MLP ---
        self.numeric_net = nn.Sequential(
            nn.Linear(num_dim, numeric_hidden),
            nn.BatchNorm1d(numeric_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(numeric_hidden, numeric_hidden),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )

        # --- Stream 3: Flag MLP (deliberately shallow) ---
        self.flag_net = nn.Sequential(
            nn.Linear(flag_dim, flag_hidden),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
        )

        # --- Fusion head ---
        fusion_input = self.bert_hidden + numeric_hidden + flag_hidden
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input, fusion_hidden),
            nn.BatchNorm1d(fusion_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, fusion_hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(fusion_hidden // 2, 1),  # raw logit
        )

    def forward(self, input_ids, attention_mask, numeric, flags):
        # Stream 1: FinBERT → [CLS] token
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = bert_out.last_hidden_state[:, 0, :]  # [batch, 768]
        cls_emb = self.text_dropout(cls_emb)

        # Stream 2: Numeric
        num_emb = self.numeric_net(numeric)  # [batch, 64]

        # Stream 3: Flags
        flag_emb = self.flag_net(flags)  # [batch, 32]

        # Fusion
        fused = torch.cat([cls_emb, num_emb, flag_emb], dim=1)  # [batch, 864]
        logit = self.fusion_head(fused)  # [batch, 1]

        return logit.squeeze(-1)  # [batch]

# --- Instantiate models ---
num_dim = len(NUM_COLS)
flag_dim = len(FLAG_COLS)

# Baseline model (same architecture, will be trained with standard loss)
baseline_model = HybridLoanModel(
    num_dim=num_dim,
    flag_dim=flag_dim,
    freeze_bert=True,
    dropout=0.3,
).to(DEVICE)

# Regulatory model (same architecture, will be trained with guardrailed loss)
regulatory_model = HybridLoanModel(
    num_dim=num_dim,
    flag_dim=flag_dim,
    freeze_bert=True,
    dropout=0.3,
).to(DEVICE)

print(f"\nModels on: {DEVICE}")
total_params = sum(p.numel() for p in baseline_model.parameters())
trainable_params = sum(p.numel() for p in baseline_model.parameters() if p.requires_grad)
print(f"Total params:     {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Frozen params:    {total_params - trainable_params:,}")

# Quick sanity check with one batch
sample_batch = next(iter(train_loader))
with torch.no_grad():
    sample_logits = baseline_model(
        input_ids=sample_batch["input_ids"].to(DEVICE),
        attention_mask=sample_batch["attention_mask"].to(DEVICE),
        numeric=sample_batch["numeric"].to(DEVICE),
        flags=sample_batch["flags"].to(DEVICE),
    )
print(f"\nSanity check — output shape: {sample_logits.shape}")
print(f"Sample logits: {sample_logits[:5].cpu().numpy()}")
print("Block 3 complete.")

Loaded — Train: 35,000  Val: 7,500  Test: 7,500

POST-RELOAD SCALE CHECK (means ~0, stds ~1):
        annual_inc           dti     loan_amnt  loan_to_income    income_log  \
mean  2.683821e-16  1.013031e-16 -1.258676e-17    8.932537e-18  4.981311e-15   
std   1.000014e+00  1.000014e+00  1.000014e+00    1.000014e+00  1.000014e+00   
min  -1.450208e+00 -1.421669e+00 -1.605181e+00   -4.393298e-01 -1.926363e+01   
max   3.186577e+01  7.611515e+01  2.919659e+00    9.909860e+01  6.089306e+00   

          loan_log  monthly_income  monthly_debt_est  monthly_payment_est  
mean -2.394529e-16   -8.567115e-17     -2.395544e-17        -1.720528e-16  
std   1.000014e+00    1.000014e+00      1.000014e+00         1.000014e+00  
min  -4.540928e+00   -1.450208e+00     -1.435753e+00        -1.605181e+00  
max   1.757047e+00    3.186577e+01      5.580750e+01         2.919659e+00  


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  FinBERT: PARTIALLY FROZEN (layers 10-11 + pooler trainable)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  FinBERT: PARTIALLY FROZEN (layers 10-11 + pooler trainable)

Models on: cuda
Total params:     109,606,721
Trainable params: 14,890,817
Frozen params:    94,715,904

Sanity check — output shape: torch.Size([64])
Sample logits: [-0.00819899  0.1726101   0.11748005  0.07655631  0.2761406 ]
Block 3 complete.


In [6]:
for name, df in [("train", train_df), ("val", val_df)]:
    print(f"\n--- {name} ---")
    print("NaN counts:\n", df[NUM_COLS + FLAG_COLS + ["label", "violation_score"]].isna().sum())
    print("Inf counts:\n", np.isinf(df[NUM_COLS + FLAG_COLS].values).sum(axis=0))
    print("violation_score range:", df["violation_score"].min(), "→", df["violation_score"].max())
    print("label unique:", df["label"].unique())


--- train ---
NaN counts:
 annual_inc             0
dti                    0
loan_amnt              0
loan_to_income         0
income_log             0
loan_log               0
monthly_income         0
monthly_debt_est       0
monthly_payment_est    0
flag_dti_43            0
flag_lti_high          0
flag_pti_high          0
flag_predatory         0
flag_dual_stress       0
flag_debt_burden       0
flag_large_loan_dti    0
label                  0
violation_score        0
dtype: int64
Inf counts:
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
violation_score range: 0.0 → 1.0
label unique: [0 1]

--- val ---
NaN counts:
 annual_inc             0
dti                    0
loan_amnt              0
loan_to_income         0
income_log             0
loan_log               0
monthly_income         0
monthly_debt_est       0
monthly_payment_est    0
flag_dti_43            0
flag_lti_high          0
flag_pti_high          0
flag_predatory         0
flag_dual_stress       0
flag_debt_burden       0
flag_lar

### Feature engineering and scaling

In [ ]:
# ============================================================
# BLOCK 4: LOSS FUNCTIONS AND TRAINING LOOP
# ============================================================

# Compute pos_weight from training labels (ratio of negatives to positives)
pos_weight = torch.tensor(
    [(train_df['label'] == 0).sum() / (train_df['label'] == 1).sum()],
    dtype=torch.float32
).to(DEVICE)

print(f"pos_weight: {pos_weight.item():.4f}")


# ============================================================
# LOSS FUNCTIONS
# ============================================================

def baseline_loss_fn(logits, labels):
    return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)


def regulatory_loss_fn(logits, labels, violation_scores, lambda_reg=LAMBDA_REG, tau=TAU_TRAIN):
    bce = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=pos_weight)

    probs = torch.sigmoid(logits)
    soft_penalty = F.relu(tau - probs)
    weighted_penalty = violation_scores * soft_penalty
    reg_penalty = weighted_penalty.mean()

    total_loss = bce + lambda_reg * reg_penalty
    return total_loss, bce.item(), reg_penalty.item()


# ============================================================
# TRAINING FUNCTION (shared by both models)
# ============================================================

def train_model(
    model,
    train_loader,
    val_loader,
    loss_type="baseline",  # "baseline" or "regulatory"
    epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    patience=3,
):
    """
    Trains a HybridLoanModel with either baseline or regulatory loss.
    """
    optimizer = torch.optim.AdamW([
        {"params": list(filter(lambda p: p.requires_grad, model.bert.parameters())), "lr": 5e-6},
        {"params": model.numeric_net.parameters(), "lr": lr},
        {"params": model.flag_net.parameters(), "lr": lr},
        {"params": model.fusion_head.parameters(), "lr": lr},
    ], weight_decay=1e-2)

    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader)  # 1 epoch warmup

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "train_bce": [], "train_reg": []}

    print(f"\n{'='*60}")
    print(f"Training: {loss_type.upper()} model")
    print(f"Epochs: {epochs} | LR: {lr} | Patience: {patience}")
    if loss_type == "regulatory":
        print(f"Lambda: {LAMBDA_REG} | Tau: {TAU_TRAIN}")
    print(f"{'='*60}")

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        epoch_bce = 0.0
        epoch_reg = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch in pbar:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            numeric = batch["numeric"].to(DEVICE)
            flags = batch["flags"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            violation_scores = batch["violation_score"].to(DEVICE)

            optimizer.zero_grad()

            logits = model(input_ids, attention_mask, numeric, flags)

            if loss_type == "regulatory":
                loss, bce_val, reg_val = regulatory_loss_fn(
                    logits, labels, violation_scores)
                epoch_bce += bce_val
                epoch_reg += reg_val
            else:
                loss = baseline_loss_fn(logits, labels)
                epoch_bce += loss.item()
                epoch_reg += 0.0

            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                max_norm=1.0,
            )
            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            num_batches += 1

            pbar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "lr": f"{scheduler.get_last_lr()[0]:.2e}",
            })

        # --- Epoch averages ---
        avg_train_loss = epoch_loss / num_batches
        avg_bce = epoch_bce / num_batches
        avg_reg = epoch_reg / num_batches

        history["train_loss"].append(avg_train_loss)
        history["train_bce"].append(avg_bce)
        history["train_reg"].append(avg_reg)

        # --- Validation ---
        model.eval()
        val_loss_total = 0.0
        val_batches = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                numeric = batch["numeric"].to(DEVICE)
                flags = batch["flags"].to(DEVICE)
                labels = batch["label"].to(DEVICE)
                violation_scores = batch["violation_score"].to(DEVICE)

                logits = model(input_ids, attention_mask, numeric, flags)

                if loss_type == "regulatory":
                    v_loss, _, _ = regulatory_loss_fn(
                        logits, labels, violation_scores
                    )
                else:
                    v_loss = baseline_loss_fn(logits, labels)

                val_loss_total += v_loss.item()
                val_batches += 1

        avg_val_loss = val_loss_total / val_batches
        history["val_loss"].append(avg_val_loss)

        # --- Logging ---
        print(f"\n  Epoch {epoch+1}/{epochs}:")
        print(f"    Train loss: {avg_train_loss:.4f}  (BCE: {avg_bce:.4f}, Reg: {avg_reg:.4f})")
        print(f"    Val loss:   {avg_val_loss:.4f}")

        # --- Early stopping ---
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
            print(f"    ✓ New best val loss — checkpoint saved")
        else:
            patience_counter += 1
            print(f"    ✗ No improvement ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"    ⚠ Early stopping triggered at epoch {epoch+1}")
                break

    # --- Restore best weights ---
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\n  Restored best checkpoint (val loss: {best_val_loss:.4f})")

    return model, history


# ============================================================
# TRAIN BOTH MODELS
# ============================================================

# --- Train baseline ---
baseline_model, baseline_history = train_model(
    baseline_model,
    train_loader,
    val_loader,
    loss_type="baseline",
    epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    patience=3,
)

# --- Train regulatory ---
regulatory_model, regulatory_history = train_model(
    regulatory_model,
    train_loader,
    val_loader,
    loss_type="regulatory",
    epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    patience=3,
)

# ============================================================
# TRAINING CURVES PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Baseline curves
ax = axes[0]
ax.plot(baseline_history["train_loss"], label="Train Loss", marker="o")
ax.plot(baseline_history["val_loss"], label="Val Loss", marker="s")
ax.set_title("Baseline Model")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# Regulatory curves
ax = axes[1]
ax.plot(regulatory_history["train_loss"], label="Train Loss (total)", marker="o")
ax.plot(regulatory_history["val_loss"], label="Val Loss (total)", marker="s")
ax.plot(regulatory_history["train_bce"], label="Train BCE", marker="^", linestyle="--", alpha=0.7)
ax.plot(regulatory_history["train_reg"], label="Train Reg Penalty", marker="v", linestyle="--", alpha=0.7)
ax.set_title("Regulatory Model")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved.")

# --- Save models ---
torch.save(baseline_model.state_dict(), "baseline_model.pt")
torch.save(regulatory_model.state_dict(), "regulatory_model.pt")
print("Models saved: baseline_model.pt, regulatory_model.pt")

print("\nBlock 4 complete.")

pos_weight: 4.0093

Training: BASELINE model
Epochs: 5 | LR: 2e-05 | Patience: 3


Epoch 1/5: 100%|██████████| 547/547 [04:28<00:00,  2.04it/s, loss=1.0468, lr=5.00e-06]



  Epoch 1/5:
    Train loss: 1.1080  (BCE: 1.1080, Reg: 0.0000)
    Val loss:   1.0907
    ✓ New best val loss — checkpoint saved


Epoch 2/5: 100%|██████████| 547/547 [04:48<00:00,  1.90it/s, loss=0.9195, lr=3.75e-06]



  Epoch 2/5:
    Train loss: 1.0548  (BCE: 1.0548, Reg: 0.0000)
    Val loss:   1.0136
    ✓ New best val loss — checkpoint saved


...

Epoch 2/5: 100%|██████████| 547/547 [04:47<00:00,  1.90it/s, loss=0.8832, lr=3.75e-06]



  Epoch 2/5:
    Train loss: 1.0435  (BCE: 1.0435, Reg: 0.0001)
    Val loss:   1.0193
    ✓ New best val loss — checkpoint saved


Epoch 3/5: 100%|██████████| 547/547 [04:47<00:00,  1.90it/s, loss=0.9691, lr=2.50e-06]



  Epoch 3/5:
    Train loss: 1.0303  (BCE: 1.0302, Reg: 0.0002)
    Val loss:   1.0187
    ✓ New best val loss — checkpoint saved


Epoch 4/5:  84%|████████▍ | 462/547 [04:02<00:44,  1.91it/s, loss=1.3678, lr=1.44e-06]

### Tokenise and build HF datasets

In [ ]:
# --- Load best saved weights ---
baseline_model.load_state_dict(torch.load("baseline_model.pt", map_location=DEVICE))
regulatory_model.load_state_dict(torch.load("regulatory_model.pt", map_location=DEVICE))
baseline_model.eval()
regulatory_model.eval()
print("Loaded best weights for baseline and regulatory models.")

# ============================================================
# BLOCK 5: INFERENCE, TEMPERATURE SCALING, AND METRICS
# ============================================================

# ============================================================
# PART A: RAW INFERENCE ON TEST SET
# ============================================================

def collect_predictions(model, loader, device=DEVICE):
    """
    Runs inference and returns raw logits, probabilities, labels,
    and violation scores for the entire dataset.
    """
    model.eval()
    all_logits = []
    all_labels = []
    all_violations = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Inference"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            numeric = batch["numeric"].to(device)
            flags = batch["flags"].to(device)

            logits = model(input_ids, attention_mask, numeric, flags)

            all_logits.append(logits.cpu())
            all_labels.append(batch["label"])
            all_violations.append(batch["violation_score"])

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    violations = torch.cat(all_violations)
    probs = torch.sigmoid(logits)

    return logits, probs, labels, violations


print("Collecting baseline predictions...")
base_logits, base_probs, base_labels, base_violations = collect_predictions(
    baseline_model, test_loader
)

print("Collecting regulatory predictions...")
reg_logits, reg_probs, reg_labels, reg_violations = collect_predictions(
    regulatory_model, test_loader
)

# Convert to numpy for sklearn metrics
base_probs_np = base_probs.numpy()
reg_probs_np = reg_probs.numpy()
labels_np = base_labels.numpy()
violations_np = base_violations.numpy()

print(f"Test samples: {len(labels_np):,}")
print(f"Risky ratio:  {labels_np.mean():.4f}")


# ============================================================
# PART B: TEMPERATURE SCALING (POST-HOC CALIBRATION)
# ============================================================

class TemperatureScaler(nn.Module):
    """
    Learns a single scalar temperature T to calibrate logits.
    calibrated_prob = sigmoid(logit / T)

    T > 1 → softens predictions (less overconfident)
    T < 1 → sharpens predictions
    T = 1 → no change
    """
    def __init__(self, init_temp=1.5):
        super().__init__()
        self.temperature = nn.Parameter(torch.tensor(float(init_temp)))

    def forward(self, logits):
        return logits / self.temperature

    def calibrated_probs(self, logits):
        return torch.sigmoid(self.forward(logits))


def fit_temperature(logits, labels, lr=0.01, max_iter=300):
    """
    Fits temperature on a held-out set (validation) by minimizing NLL.
    Returns the fitted TemperatureScaler.
    """
    scaler_ts = TemperatureScaler(init_temp=1.5)
    optimizer = torch.optim.LBFGS([scaler_ts.temperature], lr=lr, max_iter=max_iter)

    def closure():
        optimizer.zero_grad()
        scaled_logits = scaler_ts(logits)
        loss = F.binary_cross_entropy_with_logits(scaled_logits, labels)
        loss.backward()
        return loss

    optimizer.step(closure)

    print(f"  Fitted temperature: {scaler_ts.temperature.item():.4f}")
    return scaler_ts


# --- Collect validation logits for temperature fitting ---
print("\nCollecting validation logits for temperature fitting...")
base_val_logits, _, base_val_labels, _ = collect_predictions(baseline_model, val_loader)
reg_val_logits, _, reg_val_labels, _ = collect_predictions(regulatory_model, val_loader)

# --- Fit temperature on validation set ---
print("\nFitting temperature — Baseline:")
base_temp_scaler = fit_temperature(base_val_logits, base_val_labels)

print("Fitting temperature — Regulatory:")
reg_temp_scaler = fit_temperature(reg_val_logits, reg_val_labels)

# --- Apply calibration to test logits ---
with torch.no_grad():
    base_probs_cal = base_temp_scaler.calibrated_probs(base_logits).numpy()
    reg_probs_cal = reg_temp_scaler.calibrated_probs(reg_logits).numpy()

print(f"\nCalibrated probability stats:")
print(f"  Baseline  — mean: {base_probs_cal.mean():.4f}, std: {base_probs_cal.std():.4f}")
print(f"  Regulatory — mean: {reg_probs_cal.mean():.4f}, std: {reg_probs_cal.std():.4f}")


# ============================================================
# PART C: FULL METRICS SUITE
# ============================================================

def compute_full_metrics(probs, probs_cal, labels, violations, model_name, threshold=TAU_DECISION):
    """
    Computes comprehensive metrics for one model.
    Uses CALIBRATED probabilities for calibration metrics,
    and RAW probabilities for classification decisions.
    """
    # Binary predictions at operating threshold
    preds = (probs >= threshold).astype(int)

    # --- Classification metrics ---
    acc = accuracy_score(labels, preds)
    bal_acc = balanced_accuracy_score(labels, preds)
    prec = precision_score(labels, preds, zero_division=0)
    rec = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)
    spec = recall_score(labels, preds, pos_label=0, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    # --- Ranking metrics (threshold-independent) ---
    roc_auc = roc_auc_score(labels, probs)
    pr_auc = average_precision_score(labels, probs)

    # --- Calibration metrics (use calibrated probs) ---
    brier = brier_score_loss(labels, probs_cal)

    # Expected Calibration Error (ECE)
    n_bins = 10
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs_cal >= bin_boundaries[i]) & (probs_cal < bin_boundaries[i + 1])
        if mask.sum() > 0:
            bin_acc = labels[mask].mean()
            bin_conf = probs_cal[mask].mean()
            ece += mask.sum() * abs(bin_acc - bin_conf)
    ece /= len(labels)

    # --- Regulatory metrics ---
    # False low-risk rate: flagged borrowers predicted safe
    flagged_mask = violations > 0
    if flagged_mask.sum() > 0:
        flagged_preds = preds[flagged_mask]
        flagged_labels = labels[flagged_mask]
        false_safe_rate = (flagged_preds == 0).mean()  # predicted safe among flagged
        flagged_recall = recall_score(flagged_labels, flagged_preds, zero_division=0)
    else:
        false_safe_rate = 0.0
        flagged_recall = 0.0

    # Compliance breach rate: risky borrowers (label=1) with violations predicted safe
    risky_flagged = (labels == 1) & (violations > 0)
    if risky_flagged.sum() > 0:
        breach_rate = (preds[risky_flagged] == 0).mean()
    else:
        breach_rate = 0.0

    metrics = {
        "Model": model_name,
        "Threshold": threshold,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "Precision": prec,
        "Recall (Sensitivity)": rec,
        "Specificity": spec,
        "F1 Score": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Brier Score (cal)": brier,
        "ECE (cal)": ece,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "False Safe Rate (flagged)": false_safe_rate,
        "Flagged Recall": flagged_recall,
        "Compliance Breach Rate": breach_rate,
    }

    return metrics


# --- Compute metrics for both models ---
print(f"\nComputing metrics at DECISION TAU = {TAU_DECISION}...\n")

base_metrics = compute_full_metrics(
    base_probs_np, base_probs_cal, labels_np, violations_np,
    model_name="Baseline",
    threshold=TAU_DECISION,
)

reg_metrics = compute_full_metrics(
    reg_probs_np, reg_probs_cal, labels_np, violations_np,
    model_name="Regulatory",
    threshold=TAU_DECISION,
)

# --- Display as table ---
metrics_df = pd.DataFrame([base_metrics, reg_metrics]).set_index("Model").T
print(metrics_df.to_string())

# --- Save ---
metrics_df.to_csv("clean_performance_table.csv")
print("\nSaved: clean_performance_table.csv")


# ============================================================
# PART D: COMPLIANCE TABLE
# ============================================================

def compute_compliance_table(probs, labels, violations, model_name, threshold=TAU_DECISION):
    """Detailed compliance breakdown by violation severity bucket."""
    preds = (probs >= threshold).astype(int)

    rows = []
    # Overall
    flagged = violations > 0
    if flagged.sum() > 0:
        rows.append({
            "Model": model_name,
            "Bucket": "All Flagged (v>0)",
            "N": int(flagged.sum()),
            "Predicted Safe (%)": (preds[flagged] == 0).mean() * 100,
            "Recall": recall_score(labels[flagged], preds[flagged], zero_division=0),
        })

    # By severity quartile
    for q_label, (lo, hi) in [
        ("Low (0-0.25)", (0.01, 0.25)),
        ("Medium (0.25-0.5)", (0.25, 0.5)),
        ("High (0.5-0.75)", (0.5, 0.75)),
        ("Critical (0.75-1.0)", (0.75, 1.01)),
    ]:
        mask = (violations >= lo) & (violations < hi)
        if mask.sum() > 0:
            rows.append({
                "Model": model_name,
                "Bucket": q_label,
                "N": int(mask.sum()),
                "Predicted Safe (%)": (preds[mask] == 0).mean() * 100,
                "Recall": recall_score(labels[mask], preds[mask], zero_division=0),
            })

    return rows


compliance_rows = []
compliance_rows += compute_compliance_table(base_probs_np, labels_np, violations_np, "Baseline", TAU_DECISION)
compliance_rows += compute_compliance_table(reg_probs_np, labels_np, violations_np, "Regulatory", TAU_DECISION)

compliance_df = pd.DataFrame(compliance_rows)
print("\n--- Compliance Breakdown ---")
print(compliance_df.to_string(index=False))
compliance_df.to_csv("compliance_table.csv", index=False)
print("Saved: compliance_table.csv")


# ============================================================
# PART E: THRESHOLD SWEEP TABLE
# ============================================================

def threshold_sweep(probs, labels, violations, model_name, thresholds=None):
    """Sweep across thresholds to find optimal operating point."""
    if thresholds is None:
        thresholds = np.arange(0.30, 0.75, 0.05)

    rows = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        flagged = violations > 0

        rec = recall_score(labels, preds, zero_division=0)
        spec = recall_score(labels, preds, pos_label=0, zero_division=0)
        f1 = f1_score(labels, preds, zero_division=0)
        false_safe = (preds[flagged] == 0).mean() if flagged.sum() > 0 else 0.0

        rows.append({
            "Model": model_name,
            "Threshold": round(t, 2),
            "Recall": rec,
            "Specificity": spec,
            "F1": f1,
            "False Safe Rate (flagged)": false_safe,
        })

    return rows


sweep_rows = []
sweep_rows += threshold_sweep(base_probs_np, labels_np, violations_np, "Baseline")
sweep_rows += threshold_sweep(reg_probs_np, labels_np, violations_np, "Regulatory")

sweep_df = pd.DataFrame(sweep_rows)
print("\n--- Threshold Sweep ---")
print(sweep_df.to_string(index=False))
sweep_df.to_csv("threshold_sweep_table.csv", index=False)
print("Saved: threshold_sweep_table.csv")


# ============================================================
# PART F: VIOLATION SCORE TABLE
# ============================================================

def violation_score_table(probs, labels, violations, model_name):
    """Mean predicted risk by violation score bucket."""
    rows = []
    for lo, hi, label in [
        (0.0, 0.001, "None (0)"),
        (0.001, 0.25, "Low (0-0.25)"),
        (0.25, 0.5, "Medium (0.25-0.5)"),
        (0.5, 0.75, "High (0.5-0.75)"),
        (0.75, 1.01, "Critical (0.75-1.0)"),
    ]:
        mask = (violations >= lo) & (violations < hi)
        if mask.sum() > 0:
            rows.append({
                "Model": model_name,
                "Violation Bucket": label,
                "N": int(mask.sum()),
                "Mean P(Risky)": probs[mask].mean(),
                "Actual Risky (%)": labels[mask].mean() * 100,
            })
    return rows


vs_rows = []
vs_rows += violation_score_table(base_probs_np, labels_np, violations_np, "Baseline")
vs_rows += violation_score_table(reg_probs_np, labels_np, violations_np, "Regulatory")

vs_df = pd.DataFrame(vs_rows)
print("\n--- Violation Score Table ---")
print(vs_df.to_string(index=False))
vs_df.to_csv("violation_score_table.csv", index=False)
print("Saved: violation_score_table.csv")


# ============================================================
# PART G: SAVE PREDICTIONS FOR LATER BLOCKS
# ============================================================

test_predictions = pd.DataFrame({
    "label": labels_np,
    "violation_score": violations_np,
    "base_logit": base_logits.numpy(),
    "base_prob_raw": base_probs_np,
    "base_prob_cal": base_probs_cal,
    "reg_logit": reg_logits.numpy(),
    "reg_prob_raw": reg_probs_np,
    "reg_prob_cal": reg_probs_cal,
})
test_predictions.to_csv("test_predictions.csv", index=False)
print("\nSaved: test_predictions.csv")

# Save temperature scalers
torch.save(base_temp_scaler.state_dict(), "base_temp_scaler.pt")
torch.save(reg_temp_scaler.state_dict(), "reg_temp_scaler.pt")
print("Saved: base_temp_scaler.pt, reg_temp_scaler.pt")

print("\nBlock 5 complete.")

### Model, collator, trainer definitions

In [ ]:
# ============================================================
# BLOCK 6: VISUALIZATION SUITE
# ============================================================

# --- Load saved predictions ---
pred_df = pd.read_csv("test_predictions.csv")

labels_np = pred_df["label"].values
violations_np = pred_df["violation_score"].values
base_probs_np = pred_df["base_prob_raw"].values
reg_probs_np = pred_df["reg_prob_raw"].values
base_probs_cal = pred_df["base_prob_cal"].values
reg_probs_cal = pred_df["reg_prob_cal"].values

base_preds = (base_probs_np >= TAU_DECISION).astype(int)
reg_preds = (reg_probs_np >= TAU_DECISION).astype(int)

print(f"Loaded {len(pred_df):,} test predictions.")


# ============================================================
# PLOT 1: CONFUSION MATRICES (side by side)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, title in [
    (axes[0], base_preds, "Baseline Model"),
    (axes[1], reg_preds, "Regulatory Model"),
]:
    cm = confusion_matrix(labels_np, preds)
    sns.heatmap(
        cm, annot=True, fmt=",d", cmap="Blues", ax=ax,
        xticklabels=["Low Risk", "Risky"],
        yticklabels=["Low Risk", "Risky"],
        annot_kws={"size": 14},
    )
    ax.set_title(f"{title} (τ={TAU_DECISION})", fontsize=13)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices.png")


# ============================================================
# PLOT 2: ROC, PR, AND CALIBRATION CURVES
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- ROC Curve ---
ax = axes[0]
for probs, name, color in [
    (base_probs_np, "Baseline", "#1f77b4"),
    (reg_probs_np, "Regulatory", "#d62728"),
]:
    fpr, tpr, _ = roc_curve(labels_np, probs)
    auc_val = roc_auc_score(labels_np, probs)
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.4f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5)
ax.set_title("ROC Curve", fontsize=13)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

# --- PR Curve ---
ax = axes[1]
for probs, name, color in [
    (base_probs_np, "Baseline", "#1f77b4"),
    (reg_probs_np, "Regulatory", "#d62728"),
]:
    prec_vals, rec_vals, _ = precision_recall_curve(labels_np, probs)
    ap = average_precision_score(labels_np, probs)
    ax.plot(rec_vals, prec_vals, label=f"{name} (AP={ap:.4f})", color=color, linewidth=2)

prevalence = labels_np.mean()
ax.axhline(y=prevalence, linestyle="--", color="gray", alpha=0.5, label=f"Prevalence={prevalence:.3f}")
ax.set_title("Precision-Recall Curve", fontsize=13)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)

# --- Calibration Curve (uses CALIBRATED probabilities) ---
ax = axes[2]
for probs_c, name, color in [
    (base_probs_cal, "Baseline", "#1f77b4"),
    (reg_probs_cal, "Regulatory", "#d62728"),
]:
    prob_true, prob_pred = calibration_curve(labels_np, probs_c, n_bins=10, strategy="quantile")
    brier = brier_score_loss(labels_np, probs_c)
    ax.plot(prob_pred, prob_true, marker="o", label=f"{name} (Brier={brier:.4f})", color=color, linewidth=2)

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.5, label="Perfect calibration")
ax.set_title("Calibration Curve (Temperature-Scaled)", fontsize=13)
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Observed Positive Rate")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("roc_pr_calibration.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: roc_pr_calibration.png")


# ============================================================
# PLOT 3: THRESHOLD SWEEP — RECALL, SPECIFICITY, FALSE SAFE RATE
# ============================================================

sweep_df = pd.read_csv("threshold_sweep_table.csv")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric, ylabel in [
    (axes[0], "Recall", "Recall (Sensitivity)"),
    (axes[1], "Specificity", "Specificity"),
    (axes[2], "False Safe Rate (flagged)", "False Safe Rate (flagged borrowers)"),
]:
    for name, color, ls in [
        ("Baseline", "#1f77b4", "-"),
        ("Regulatory", "#d62728", "-"),
    ]:
        subset = sweep_df[sweep_df["Model"] == name]
        ax.plot(
            subset["Threshold"], subset[metric],
            label=name, color=color, linestyle=ls,
            marker="o", linewidth=2,
        )

    # Mark operating threshold
    ax.axvline(x=TAU_DECISION, linestyle="--", color="green", alpha=0.7, label=f"TAU_DECISION={TAU_DECISION}")
    ax.set_title(metric, fontsize=13)
    ax.set_xlabel("Decision Threshold")
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("threshold_sweep_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: threshold_sweep_plot.png")


# ============================================================
# PLOT 4: VIOLATION SCORE vs MEAN P(RISKY)
# ============================================================

vs_df = pd.read_csv("violation_score_table.csv")

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(vs_df[vs_df["Model"] == "Baseline"]))
width = 0.35

base_vals = vs_df[vs_df["Model"] == "Baseline"]["Mean P(Risky)"].values
reg_vals = vs_df[vs_df["Model"] == "Regulatory"]["Mean P(Risky)"].values
bucket_labels = vs_df[vs_df["Model"] == "Baseline"]["Violation Bucket"].values
actual_risky = vs_df[vs_df["Model"] == "Baseline"]["Actual Risky (%)"].values / 100.0

ax.bar(x - width / 2, base_vals, width, label="Baseline", color="#1f77b4", alpha=0.8)
ax.bar(x + width / 2, reg_vals, width, label="Regulatory", color="#d62728", alpha=0.8)
ax.plot(x, actual_risky, "k^--", label="Actual Risky Rate", markersize=10, linewidth=2)

ax.axhline(y=TAU_DECISION, linestyle=":", color="green", alpha=0.7, label=f"TAU_DECISION={TAU_DECISION}")
ax.set_xlabel("Violation Severity Bucket", fontsize=12)
ax.set_ylabel("Mean P(Risky)", fontsize=12)
ax.set_title("Predicted Risk by Violation Severity", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(bucket_labels, rotation=15, ha="right")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("violation_score_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: violation_score_plot.png")


# ============================================================
# PLOT 5: PROBABILITY DISTRIBUTION — BASELINE vs REGULATORY
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, probs, name, color in [
    (axes[0], base_probs_np, "Baseline", "#1f77b4"),
    (axes[1], reg_probs_np, "Regulatory", "#d62728"),
]:
    # Safe loans
    ax.hist(
        probs[labels_np == 0], bins=50, alpha=0.6,
        label="Safe (label=0)", color="green", density=True,
    )
    # Risky loans
    ax.hist(
        probs[labels_np == 1], bins=50, alpha=0.6,
        label="Risky (label=1)", color="red", density=True,
    )
    ax.axvline(x=TAU_DECISION, linestyle="--", color="black", linewidth=2, label=f"TAU_DECISION={TAU_DECISION}")
    ax.set_title(f"{name} — P(Risky) Distribution", fontsize=13)
    ax.set_xlabel("P(Risky)")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("probability_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: probability_distributions.png")


# ============================================================
# PLOT 6: COMPLIANCE HEATMAP — FALSE SAFE RATE BY SEVERITY
# ============================================================

compliance_df = pd.read_csv("compliance_table.csv")

# Pivot for heatmap
pivot = compliance_df.pivot(index="Bucket", columns="Model", values="Predicted Safe (%)")

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pivot, annot=True, fmt=".1f", cmap="YlOrRd_r", ax=ax,
    linewidths=0.5, cbar_kws={"label": "Predicted Safe (%)"},
)
ax.set_title(f"False Safe Rate by Violation Severity (τ={TAU_DECISION})", fontsize=13)
ax.set_ylabel("Violation Severity Bucket")
ax.set_xlabel("")

plt.tight_layout()
plt.savefig("compliance_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: compliance_heatmap.png")


print("\nBlock 6 complete. All plots saved.")

### Hyperparameter Optimization

In [ ]:
# ============================================================
# BLOCK 7: ADVERSARIAL TEXT ATTACK & ROBUSTNESS EVALUATION
# ============================================================

import random

# ============================================================
# 7A: ADVERSARIAL TEXT GENERATOR
# ============================================================

ADVERSARIAL_TEMPLATES = [
    "I have an excellent track record of repaying all my loans on time. "
    "My financial situation is very stable and I am confident in my ability to repay.",

    "I am a responsible borrower with a long history of on-time payments. "
    "This loan will be used to consolidate debt and improve my financial standing.",

    "My income has been steadily increasing and I have significant savings. "
    "I have never missed a payment and my credit history is impeccable.",

    "I work in a very stable industry with guaranteed income growth. "
    "This loan is a small fraction of my annual earnings and poses no risk.",

    "I am requesting this loan to invest in my education which will significantly "
    "increase my earning potential. I have zero outstanding debts.",

    "As a senior professional with over 15 years of experience, my job security "
    "is excellent. I have multiple income streams and substantial assets.",

    "My debt-to-income ratio is well within healthy limits and I maintain "
    "an emergency fund covering 12 months of expenses. Repayment is guaranteed.",

    "I have been pre-approved by multiple lenders due to my outstanding credit. "
    "This loan will be fully repaid within the first year from my bonus alone.",

    "I own my home outright with no mortgage. My monthly obligations are minimal "
    "and this loan payment would represent less than 5% of my disposable income.",

    "My employer provides guaranteed annual bonuses that alone cover this loan. "
    "I have a perfect repayment history across all financial obligations.",
]


def generate_adversarial_text(original_text: str, seed: int = None) -> str:
    if seed is not None:
        random.seed(seed)
    return random.choice(ADVERSARIAL_TEMPLATES)


# ============================================================
# 7B: BUILD PAIRED CLEAN / ADVERSARIAL DATASETS
# ============================================================

adv_eval_df = test_df.copy()
adv_eval_df["clean_text"] = adv_eval_df["text"].values.copy()
adv_eval_df["adversarial_text"] = [
    generate_adversarial_text(t, seed=i)
    for i, t in enumerate(adv_eval_df["text"])
]

# --- Clean dataset (original text) ---
clean_eval_df = adv_eval_df.copy()

# --- Adversarial dataset (swapped text, everything else identical) ---
adversarial_eval_df = adv_eval_df.copy()
adversarial_eval_df["text"] = adversarial_eval_df["adversarial_text"]

# Reuse LoanDataset from Block 3 + fitted scaler
#scaler = joblib.load(SCALER_PATH)

clean_ds = LoanDataset(clean_eval_df, tokenizer)
adv_ds = LoanDataset(adversarial_eval_df, tokenizer)

clean_loader = DataLoader(clean_ds, batch_size=64, shuffle=False)
adv_loader = DataLoader(adv_ds, batch_size=64, shuffle=False)

print(f"Paired evaluation size: {len(clean_eval_df):,} samples")
print(f"Adversarial templates: {len(ADVERSARIAL_TEMPLATES)}")
print(f"Sample adversarial text: '{adversarial_eval_df['adversarial_text'].iloc[0][:80]}...'")


# ============================================================
# 7C: INFERENCE HELPER (raw PyTorch, matches Block 4/5)
# ============================================================

@torch.no_grad()
def predict_probs(model, loader, device):
    """Return raw P(risky) from sigmoid over binary logits."""
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        numeric        = batch["numeric"].to(device)
        flags          = batch["flags"].to(device)

        logits = model(input_ids, attention_mask, numeric, flags)
        probs  = torch.sigmoid(logits).squeeze(-1)  # P(risky), shape: (batch,)
        all_probs.append(probs.cpu().numpy())

    return np.concatenate(all_probs)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Baseline model ---
base_clean_p_risky = predict_probs(baseline_model, clean_loader, device)
base_adv_p_risky = predict_probs(baseline_model, adv_loader, device)
base_clean_preds = (base_clean_p_risky >= TAU_DECISION).astype(int)
base_adv_preds = (base_adv_p_risky >= TAU_DECISION).astype(int)

# --- Regulatory model ---
reg_clean_p_risky = predict_probs(regulatory_model, clean_loader, device)
reg_adv_p_risky = predict_probs(regulatory_model, adv_loader, device)
reg_clean_preds = (reg_clean_p_risky >= TAU_DECISION).astype(int)
reg_adv_preds = (reg_adv_p_risky >= TAU_DECISION).astype(int)

labels_adv = adv_eval_df["label"].values.astype(int)
violations_adv = adv_eval_df["violation_score"].values

print("Predictions generated for both models on clean + adversarial text.")


# ============================================================
# 7D: ROBUSTNESS METRICS
# ============================================================

def attack_success_rate(clean_preds, adv_preds):
    """Fraction of samples that flipped from risky (1) to safe (0) under attack."""
    risky_mask = clean_preds == 1
    if risky_mask.sum() == 0:
        return 0.0
    flipped = ((clean_preds == 1) & (adv_preds == 0)).sum()
    return flipped / risky_mask.sum()


def build_adversarial_robustness_table(labels, violations,
                                        base_cp, base_ap, base_cpred, base_apred,
                                        reg_cp, reg_ap, reg_cpred, reg_apred):
    rows = []
    for name, cp, ap, cpred, apred in [
        ("Baseline", base_cp, base_ap, base_cpred, base_apred),
        ("Regulatory", reg_cp, reg_ap, reg_cpred, reg_apred),
    ]:
        recall_clean = recall_score(labels, cpred, zero_division=0)
        recall_adv = recall_score(labels, apred, zero_division=0)

        risky_mask = labels == 1
        false_safe_adv = (apred[risky_mask] == 0).mean() if risky_mask.sum() > 0 else 0.0

        flagged_risky = (labels == 1) & (violations > 0)
        false_safe_flagged_adv = (
            (apred[flagged_risky] == 0).mean() if flagged_risky.sum() > 0 else 0.0
        )

        rows.append({
            "Model": name,
            "Recall (clean)": recall_clean,
            "Recall (adversarial)": recall_adv,
            "Recall Drop": recall_clean - recall_adv,
            "Attack Success Rate": attack_success_rate(cpred, apred),
            "Mean Δp(risky) [adv−clean]": (ap - cp).mean(),
            "False Safe Rate (adv, risky)": false_safe_adv,
            "False Safe Rate (adv, flagged+risky)": false_safe_flagged_adv,
        })

    return pd.DataFrame(rows)


adv_table = build_adversarial_robustness_table(
    labels_adv, violations_adv,
    base_clean_p_risky, base_adv_p_risky, base_clean_preds, base_adv_preds,
    reg_clean_p_risky, reg_adv_p_risky, reg_clean_preds, reg_adv_preds,
)

print("\n— Adversarial Robustness Table —")
print(adv_table.to_string(index=False, float_format="%.4f"))
adv_table.to_csv("adversarial_robustness_table.csv", index=False)
print("Saved: adversarial_robustness_table.csv")


# ============================================================
# 7E: PAIRED PROBABILITY SHIFT ANALYSIS
# ============================================================

shift_df = pd.DataFrame({
    "label": labels_adv,
    "violation_score": violations_adv,
    "base_clean_p_risky": base_clean_p_risky,
    "base_adv_p_risky": base_adv_p_risky,
    "base_delta_p": base_adv_p_risky - base_clean_p_risky,
    "reg_clean_p_risky": reg_clean_p_risky,
    "reg_adv_p_risky": reg_adv_p_risky,
    "reg_delta_p": reg_adv_p_risky - reg_clean_p_risky,
    "base_clean_pred": base_clean_preds,
    "base_adv_pred": base_adv_preds,
    "reg_clean_pred": reg_clean_preds,
    "reg_adv_pred": reg_adv_preds,
})

shift_df.to_csv("paired_probability_shifts.csv", index=False)
print(f"\nSaved: paired_probability_shifts.csv ({len(shift_df):,} rows)")

print("\n— Probability Shift Summary —")
for name, col in [("Baseline", "base_delta_p"), ("Regulatory", "reg_delta_p")]:
    vals = shift_df[col]
    print(f"  {name}: mean={vals.mean():.5f}, std={vals.std():.5f}, "
          f"median={vals.median():.5f}, min={vals.min():.5f}, max={vals.max():.5f}")


# ============================================================
# 7F: ADVERSARIAL PROBABILITY SHIFT PLOTS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Row 1: Δp histograms ---
for ax, col, name, color in [
    (axes[0, 0], "base_delta_p", "Baseline", "#1f77b4"),
    (axes[0, 1], "reg_delta_p", "Regulatory", "#d62728"),
]:
    ax.hist(shift_df[col], bins=60, alpha=0.7, color=color, density=True, edgecolor="white")
    ax.axvline(x=0, linestyle="--", color="black", linewidth=1.5)
    ax.axvline(x=shift_df[col].mean(), linestyle="-", color="orange", linewidth=2,
               label=f"Mean={shift_df[col].mean():.4f}")
    ax.set_title(f"{name} — Δp(risky) [adversarial − clean]", fontsize=12)
    ax.set_xlabel("Δp(risky)")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(True, alpha=0.3)

# --- Row 2: Clean vs Adversarial scatter ---
for ax, cp, ap, name, color in [
    (axes[1, 0], base_clean_p_risky, base_adv_p_risky, "Baseline", "#1f77b4"),
    (axes[1, 1], reg_clean_p_risky, reg_adv_p_risky, "Regulatory", "#d62728"),
]:
    ax.scatter(cp, ap, alpha=0.1, s=5, color=color)
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1.5)
    ax.axhline(y=TAU_DECISION, linestyle=":", color="green", alpha=0.5)
    ax.axvline(x=TAU_DECISION, linestyle=":", color="green", alpha=0.5)
    ax.set_title(f"{name} — Clean vs Adversarial P(Risky)", fontsize=12)
    ax.set_xlabel("P(Risky) — Clean Text")
    ax.set_ylabel("P(Risky) — Adversarial Text")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("adversarial_shift_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: adversarial_shift_plots.png")


# ============================================================
# 7G: QUALITATIVE EXAMPLES
# ============================================================

qual_df = adv_eval_df.copy()
qual_df["base_clean_pred"] = base_clean_preds
qual_df["base_adv_pred"] = base_adv_preds
qual_df["reg_clean_pred"] = reg_clean_preds
qual_df["reg_adv_pred"] = reg_adv_preds
qual_df["base_adv_p_risky"] = base_adv_p_risky
qual_df["reg_adv_p_risky"] = reg_adv_p_risky

# Baseline fooled (1→0), Regulatory held firm
baseline_fooled = qual_df[
    (qual_df["base_clean_pred"] == 1) &
    (qual_df["base_adv_pred"] == 0) &
    (qual_df["reg_adv_pred"] == 1)
]

# Regulatory fooled, Baseline held firm
reg_fooled = qual_df[
    (qual_df["reg_clean_pred"] == 1) &
    (qual_df["reg_adv_pred"] == 0) &
    (qual_df["base_adv_pred"] == 1)
]

show_cols = [
    "violation_score", "annual_inc", "dti", "loan_amnt",
    "base_adv_p_risky", "reg_adv_p_risky", "adversarial_text",
]

print(f"\n— Baseline fooled, Regulatory correct: {len(baseline_fooled):,} cases —")
print(baseline_fooled[show_cols].head(10).to_string(index=False))

print(f"\n— Regulatory fooled, Baseline correct: {len(reg_fooled):,} cases —")
print(reg_fooled[show_cols].head(10).to_string(index=False))

print(f"\nAsymmetry ratio: {len(baseline_fooled)} : {len(reg_fooled)} "
      f"(Baseline fooled {len(baseline_fooled) - len(reg_fooled):+d} more times)")


print("\nBlock 7 complete.")

### Train baseline (lambda_reg=0.0) and save

In [ ]:
# ============================================================
# BLOCK 8: SYNTHETIC DISTRIBUTION SHIFT — PORTABILITY TEST
# ============================================================
# Purpose: Freeze both trained models and evaluate on
# systematically perturbed borrower populations to prove
# the regulatory penalty generalises beyond the training
# distribution.
# ============================================================

import copy

# ============================================================
# 8A: SHIFT DEFINITIONS
# ============================================================
# Each scenario represents a plausible macro-economic or
# demographic shift in the borrower pool.

SHIFT_SCENARIOS = {
    "Baseline (no shift)": {},

    "Recession — income ↓30%, DTI ↑40%": {
        "annual_inc": lambda x: x * 0.70,
        "dti":        lambda x: np.clip(x * 1.40, 0, 100),
    },

    "Credit boom — loan amounts ↑50%": {
        "loan_amnt": lambda x: x * 1.50,
    },

    "Subprime influx — DTI ↑60%, income ↓20%": {
        "annual_inc": lambda x: x * 0.80,
        "dti":        lambda x: np.clip(x * 1.60, 0, 100),
    },

    "High-earner pool — income ↑80%, DTI ↓30%": {
        "annual_inc": lambda x: x * 1.80,
        "dti":        lambda x: np.clip(x * 0.70, 0, 100),
    },

    "Stress test — income ↓40%, DTI ↑80%, loan ↑30%": {
        "annual_inc": lambda x: x * 0.60,
        "dti":        lambda x: np.clip(x * 1.80, 0, 100),
        "loan_amnt":  lambda x: x * 1.30,
    },
}

print(f"Defined {len(SHIFT_SCENARIOS)} shift scenarios.")


# ============================================================
# 8B: APPLY SHIFT & RECOMPUTE REGULATORY FLAGS
# ============================================================

def apply_shift(df_original, transforms: dict, scaler) -> pd.DataFrame:
    """
    Apply numeric transforms to a copy of the dataframe,
    then recompute derived features and regulatory flags.
    """
    df = df_original.copy()

    # Recover original raw numeric values from the saved scaler.
    df[NUM_COLS] = pd.DataFrame(
        scaler.inverse_transform(df[NUM_COLS]),
        columns=NUM_COLS,
        index=df.index,
    )

    orig_term_months = (
        df["loan_amnt"] / df["monthly_payment_est"].replace(0, np.nan)
    ).fillna(36.0)

    # --- Apply numeric transforms ---
    for col, fn in transforms.items():
        if col in df.columns:
            df[col] = fn(df[col].values)

    # --- Recompute derived numeric features from shifted values ---
    df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"].replace(0, np.nan)
    df["income_log"] = np.log1p(df["annual_inc"].clip(lower=0.0))
    df["loan_log"] = np.log1p(df["loan_amnt"].clip(lower=0.0))
    df["monthly_income"] = df["annual_inc"] / 12.0
    df["monthly_debt_est"] = df["monthly_income"] * (df["dti"] / 100.0)
    df["monthly_payment_est"] = df["loan_amnt"] / orig_term_months.replace(0, np.nan)

    # --- Recompute regulatory flags exactly as in Block 2 ---
    df["flag_dti_43"] = (df["dti"] > 43).astype(int)
    df["flag_lti_high"] = (df["loan_to_income"] > 0.5).astype(int)
    df["flag_pti_high"] = (df["monthly_payment_est"] / df["monthly_income"] > 0.30).astype(int)

    df["flag_predatory"] = 0
    if "int_rate" in df.columns:
        int_rate_clean = df["int_rate"].astype(str).str.replace("%", "")
        int_rate_num = pd.to_numeric(int_rate_clean, errors="coerce")
        df["flag_predatory"] = (int_rate_num > 25).astype(int).fillna(0)

    df["flag_dual_stress"] = ((df["flag_dti_43"] == 1) & (df["flag_lti_high"] == 1)).astype(int)
    df["flag_debt_burden"] = (df["monthly_debt_est"] > df["monthly_income"] * 0.50).astype(int)
    df["flag_large_loan_dti"] = (
        (df["loan_amnt"] > df["loan_amnt"].quantile(0.75)) & (df["dti"] > 30)
    ).astype(int)

    df["violation_score"] = sum(
        df[col] * weight for col, weight in VIOLATION_WEIGHTS.items()
    )
    vs_max = df["violation_score"].max()
    if vs_max > 0:
        df["violation_score"] = df["violation_score"] / vs_max

    df[NUM_COLS] = scaler.transform(df[NUM_COLS])

    return df


def predict_probs(model, loader, device=DEVICE):
    model.eval()
    probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            numeric = batch["numeric"].to(device)
            flags = batch["flags"].to(device)

            logits = model(input_ids, attention_mask, numeric, flags)
            probs.append(torch.sigmoid(logits).cpu())

    return torch.cat(probs).numpy()


print("Loaded helper functions for synthetic shift evaluation.")


# ============================================================
# 8C: INFERENCE ON SHIFTED POPULATIONS
# ============================================================

scaler = joblib.load(SCALER_PATH)
shift_results = []

for scenario_name, transforms in SHIFT_SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")

    shifted_df = apply_shift(test_df, transforms, scaler)
    shifted_ds = LoanDataset(shifted_df, tokenizer)
    shifted_loader = DataLoader(
        shifted_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )

    labels = shifted_df["label"].values.astype(int)
    violations = shifted_df["violation_score"].values

    base_probs = predict_probs(baseline_model, shifted_loader, DEVICE)
    reg_probs = predict_probs(regulatory_model, shifted_loader, DEVICE)

    base_preds = (base_probs >= TAU_DECISION).astype(int)
    reg_preds = (reg_probs >= TAU_DECISION).astype(int)

    for model_name, probs, preds in [
        ("Baseline", base_probs, base_preds),
        ("Regulatory", reg_probs, reg_preds),
    ]:
        risky_mask = labels == 1
        flagged_mask = violations > 0
        flagged_risky = risky_mask & flagged_mask

        recall = recall_score(labels, preds, zero_division=0)
        precision = precision_score(labels, preds, zero_division=0)
        f1 = f1_score(labels, preds, zero_division=0)
        bal_acc = balanced_accuracy_score(labels, preds)
        brier = brier_score_loss(labels, probs)

        false_safe = (preds[risky_mask] == 0).mean() if risky_mask.sum() > 0 else 0.0
        false_safe_flagged = (
            (preds[flagged_risky] == 0).mean() if flagged_risky.sum() > 0 else 0.0
        )
        mean_p_flagged = probs[flagged_mask].mean() if flagged_mask.sum() > 0 else np.nan

        shift_results.append({
            "Scenario": scenario_name,
            "Model": model_name,
            "Recall": recall,
            "Precision": precision,
            "F1": f1,
            "Balanced Accuracy": bal_acc,
            "Brier Score": brier,
            "False Safe Rate (risky)": false_safe,
            "False Safe Rate (flagged+risky)": false_safe_flagged,
            "Mean P(risky) flagged": mean_p_flagged,
            "N_flagged": int(flagged_mask.sum()),
            "N_total": len(labels),
        })

    print(f"  Flagged borrowers: {int(flagged_mask.sum()):,} / {len(labels):,}")
    print(f"  Baseline  — Recall: {shift_results[-2]['Recall']:.4f}, "
          f"False Safe (flagged+risky): {shift_results[-2]['False Safe Rate (flagged+risky)']:.4f}")
    print(f"  Regulatory — Recall: {shift_results[-1]['Recall']:.4f}, "
          f"False Safe (flagged+risky): {shift_results[-1]['False Safe Rate (flagged+risky)']:.4f}")


shift_results_df = pd.DataFrame(shift_results)
shift_results_df.to_csv("synthetic_shift_results.csv", index=False)
print(f"\nSaved: synthetic_shift_results.csv ({len(shift_results_df)} rows)")


# ============================================================
# 8D: SUMMARY PIVOT TABLE
# ============================================================

pivot_cols = [
    "Recall", "False Safe Rate (risky)",
    "False Safe Rate (flagged+risky)", "Mean P(risky) flagged", "Brier Score",
]

print("\n" + "=" * 70)
print("SYNTHETIC SHIFT — SUMMARY")
print("=" * 70)

for metric in pivot_cols:
    pivot = shift_results_df.pivot(
        index="Scenario", columns="Model", values=metric
    )[["Baseline", "Regulatory"]]
    pivot["Δ (Reg − Base)"] = pivot["Regulatory"] - pivot["Baseline"]
    print(f"\n--- {metric} ---")
    print(pivot.to_string(float_format="%.4f"))


# ============================================================
# 8E: VISUALISATION — REGULATORY ADVANTAGE ACROSS SHIFTS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

scenarios = shift_results_df["Scenario"].unique()
x = np.arange(len(scenarios))
width = 0.35

for ax, metric, title in [
    (axes[0], "Recall", "Recall (↑ better)"),
    (
        axes[1],
        "False Safe Rate (flagged+risky)",
        "False Safe Rate — Flagged+Risky (↓ better)",
    ),
    (
        axes[2],
        "Mean P(risky) flagged",
        "Mean P(Risky) for Flagged Borrowers (↑ better)",
    ),
]:
    base_vals = shift_results_df[shift_results_df["Model"] == "Baseline"][metric].values
    reg_vals = shift_results_df[shift_results_df["Model"] == "Regulatory"][metric].values

    ax.bar(x - width / 2, base_vals, width, label="Baseline", color="#1f77b4", alpha=0.8)
    ax.bar(x + width / 2, reg_vals, width, label="Regulatory", color="#d62728", alpha=0.8)

    ax.set_title(title, fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [s.split("—")[0].strip() if "—" in s else s[:20] for s in scenarios],
        rotation=30, ha="right", fontsize=9,
    )
    ax.legend()
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("synthetic_shift_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: synthetic_shift_comparison.png")


# ============================================================
# 8F: HEADLINE NUMBERS FOR DISCUSSION SECTION
# ============================================================

stress_df = shift_results_df[shift_results_df["Scenario"] != "Baseline (no shift)"]

for metric, direction in [
    ("Recall", "higher"),
    ("False Safe Rate (flagged+risky)", "lower"),
]:
    base_mean = stress_df[stress_df["Model"] == "Baseline"][metric].mean()
    reg_mean = stress_df[stress_df["Model"] == "Regulatory"][metric].mean()
    diff = reg_mean - base_mean
    print(f"\nAcross stress scenarios — {metric}:")
    print(f"  Baseline avg:   {base_mean:.4f}")
    print(f"  Regulatory avg: {reg_mean:.4f}")
    print(f"  Δ = {diff:+.4f} ({'✓ Regulatory ' + direction if (diff > 0) == (direction == 'higher') else '✗ Baseline ' + direction})")


print("\nBlock 8 complete.")

### Train regulatory model (lambda_reg=LAMBDA_REG) and save

In [ ]:
# ============================================================
# BLOCK 9: TEMPORAL DISTRIBUTION SHIFT — STABILITY OVER TIME
# ============================================================
# Purpose: Split the original Lending Club data by loan issue
# year, train on earlier years, and evaluate on later years
# to test whether regulatory behaviour holds as borrower
# populations evolve naturally over time.
#
# NOTE: Both models were trained on the Block 4 train split.
# Here we RE-SPLIT the held-out test set by year to see if
# regulatory advantage is stable across time periods.
# ============================================================

# ============================================================
# 9A: EXTRACT YEAR FROM TEST SET
# ============================================================

# Identify the date column (common LC names)
date_col = None
for candidate in ["issue_d", "issue_date", "earliest_cr_line", "Date"]:
    if candidate in test_df.columns:
        date_col = candidate
        break

if date_col is None:
    # Fallback: if no date column, use index-based proxy
    # (assumes data is roughly chronologically ordered)
    print("WARNING: No date column found. Using index-based temporal proxy.")
    n = len(test_df)
    test_df["_year_bucket"] = pd.cut(
        np.arange(n),
        bins=4,
        labels=["Period 1 (earliest)", "Period 2", "Period 3", "Period 4 (latest)"],
    )
    year_col = "_year_bucket"
else:
    test_df["_issue_year"] = pd.to_datetime(test_df[date_col], errors="coerce").dt.year
    year_col = "_issue_year"
    print(f"Using date column: '{date_col}'")

print(f"\nTemporal buckets in test set:")
print(test_df[year_col].value_counts().sort_index())


# ============================================================
# 9B: PER-PERIOD EVALUATION
# ============================================================

scaler = joblib.load(SCALER_PATH)
temporal_results = []

periods = sorted(test_df[year_col].dropna().unique())

for period in periods:
    print(f"\n{'='*60}")
    print(f"Period: {period}")
    print(f"{'='*60}")

    period_df = test_df[test_df[year_col] == period].copy()

    if len(period_df) < 50:
        print(f"  Skipping — only {len(period_df)} samples.")
        continue

    # Build loader
    period_ds = LoanDataset(period_df, tokenizer)
    period_loader = DataLoader(period_ds, batch_size=64, shuffle=False)

    labels = period_df["label"].values.astype(int)
    violations = period_df["violation_score"].values

    # --- Predict with both frozen models ---
    base_probs = predict_probs(baseline_model, period_loader, device)
    reg_probs = predict_probs(regulatory_model, period_loader, device)

    base_preds = (base_probs >= TAU_DECISION).astype(int)
    reg_preds = (reg_probs >= TAU_DECISION).astype(int)

    for model_name, probs, preds in [
        ("Baseline", base_probs, base_preds),
        ("Regulatory", reg_probs, reg_preds),
    ]:
        risky_mask = labels == 1
        flagged_mask = violations > 0
        flagged_risky = risky_mask & flagged_mask

        rec = recall_score(labels, preds, zero_division=0)
        prec = precision_score(labels, preds, zero_division=0)
        f1 = f1_score(labels, preds, zero_division=0)
        bal_acc = balanced_accuracy_score(labels, preds)
        brier = brier_score_loss(labels, probs)

        false_safe = (preds[risky_mask] == 0).mean() if risky_mask.sum() > 0 else 0.0
        false_safe_flagged = (
            (preds[flagged_risky] == 0).mean() if flagged_risky.sum() > 0 else 0.0
        )
        mean_p_flagged = probs[flagged_mask].mean() if flagged_mask.sum() > 0 else np.nan

        temporal_results.append({
            "Period": period,
            "Model": model_name,
            "N": len(labels),
            "N_risky": int(risky_mask.sum()),
            "N_flagged": int(flagged_mask.sum()),
            "Risky Rate": risky_mask.mean(),
            "Recall": rec,
            "Precision": prec,
            "F1": f1,
            "Balanced Accuracy": bal_acc,
            "Brier Score": brier,
            "False Safe Rate (risky)": false_safe,
            "False Safe Rate (flagged+risky)": false_safe_flagged,
            "Mean P(risky) flagged": mean_p_flagged,
        })

    print(f"  N={len(labels):,}  |  Risky rate: {risky_mask.mean():.3f}  |  "
          f"Flagged: {int(flagged_mask.sum()):,}")
    print(f"  Baseline  — Recall: {temporal_results[-2]['Recall']:.4f}, "
          f"FSR(flagged+risky): {temporal_results[-2]['False Safe Rate (flagged+risky)']:.4f}")
    print(f"  Regulatory — Recall: {temporal_results[-1]['Recall']:.4f}, "
          f"FSR(flagged+risky): {temporal_results[-1]['False Safe Rate (flagged+risky)']:.4f}")


temporal_df = pd.DataFrame(temporal_results)
temporal_df.to_csv("temporal_shift_results.csv", index=False)
print(f"\nSaved: temporal_shift_results.csv ({len(temporal_df)} rows)")


# ============================================================
# 9C: SUMMARY PIVOT TABLES
# ============================================================

print("\n" + "=" * 70)
print("TEMPORAL SHIFT — SUMMARY PIVOTS")
print("=" * 70)

for metric in [
    "Recall", "False Safe Rate (flagged+risky)",
    "Mean P(risky) flagged", "Brier Score",
]:
    pivot = temporal_df.pivot(
        index="Period", columns="Model", values=metric
    )[["Baseline", "Regulatory"]]
    pivot["Δ (Reg − Base)"] = pivot["Regulatory"] - pivot["Baseline"]
    print(f"\n--- {metric} ---")
    print(pivot.to_string(float_format="%.4f"))


# ============================================================
# 9D: TEMPORAL STABILITY PLOTS
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_metrics = [
    ("Recall", "Recall (↑ better)", axes[0, 0]),
    ("False Safe Rate (flagged+risky)", "False Safe Rate — Flagged+Risky (↓ better)", axes[0, 1]),
    ("Mean P(risky) flagged", "Mean P(Risky) for Flagged Borrowers (↑ better)", axes[1, 0]),
    ("Brier Score", "Brier Score (↓ better)", axes[1, 1]),
]

for metric, title, ax in plot_metrics:
    for model_name, color, marker in [
        ("Baseline", "#1f77b4", "o"),
        ("Regulatory", "#d62728", "s"),
    ]:
        subset = temporal_df[temporal_df["Model"] == model_name].sort_values("Period")
        ax.plot(
            range(len(subset)), subset[metric].values,
            marker=marker, color=color, linewidth=2,
            markersize=8, label=model_name,
        )

    ax.set_title(title, fontsize=12)
    ax.set_xticks(range(len(periods)))
    ax.set_xticklabels(
        [str(p) for p in sorted(temporal_df["Period"].unique())],
        rotation=30, ha="right", fontsize=9,
    )
    ax.set_xlabel("Time Period")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("temporal_shift_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: temporal_shift_comparison.png")


# ============================================================
# 9E: REGULATORY ADVANTAGE CONSISTENCY CHECK
# ============================================================

print("\n" + "=" * 70)
print("REGULATORY ADVANTAGE — CONSISTENCY ACROSS TIME")
print("=" * 70)

for metric, better in [
    ("Recall", "higher"),
    ("False Safe Rate (flagged+risky)", "lower"),
]:
    base_series = temporal_df[temporal_df["Model"] == "Baseline"].set_index("Period")[metric]
    reg_series = temporal_df[temporal_df["Model"] == "Regulatory"].set_index("Period")[metric]

    if better == "higher":
        reg_wins = (reg_series > base_series).sum()
    else:
        reg_wins = (reg_series < base_series).sum()

    total = len(base_series)

    print(f"\n{metric}:")
    print(f"  Regulatory wins {reg_wins}/{total} periods ({better} is better)")
    print(f"  Mean Baseline:   {base_series.mean():.4f}")
    print(f"  Mean Regulatory: {reg_series.mean():.4f}")
    print(f"  Mean Δ:          {(reg_series - base_series).mean():+.4f}")

    if reg_wins == total:
        print(f"  ✓ Regulatory dominates across ALL time periods.")
    elif reg_wins > total / 2:
        print(f"  ~ Regulatory wins majority of periods.")
    else:
        print(f"  ✗ Baseline competitive or better — investigate.")


# ============================================================
# 9F: POPULATION DRIFT SUMMARY (context for discussion)
# ============================================================

print("\n" + "=" * 70)
print("POPULATION DRIFT ACROSS PERIODS")
print("=" * 70)

drift_cols = ["Risky Rate", "N", "N_flagged"]
drift_summary = temporal_df[temporal_df["Model"] == "Baseline"][
    ["Period"] + drift_cols
].set_index("Period")
drift_summary["Flagged Rate"] = drift_summary["N_flagged"] / drift_summary["N"]

print(drift_summary.to_string(float_format="%.4f"))
print("\nThis shows how the borrower population naturally shifts over time,")
print("providing context for whether regulatory advantage holds under drift.")


print("\nBlock 9 complete.")

### Build adversarial test set (uses real unscaled values)

In [ ]:
# Reload test df unscaled for readable adversarial text
test_df_raw = prepare_df(pd.read_csv(TEST_PATH))  # no scaling applied

adv_df = test_df_raw[
    (test_df_raw["reg_violation"] == 1) &
    (test_df_raw["label"] == 1)
].reset_index(drop=True).copy()

print(f"Adversarial base cases: {len(adv_df)}")

safe_purposes = [
    "debt consolidation", "home improvement", "educational expenses",
    "medical bills", "small business investment"
]

safe_titles = [
    "stable income borrower", "long-term financial planning",
    "responsible credit user", "low risk applicant",
    "consistent payment history"
]

safe_emp_titles = [
    "senior manager", "government employee", "tenured professor",
    "registered nurse", "software engineer"
]

safe_openers = [
    "This is a routine loan request from a financially stable applicant.",
    "Applicant has a strong repayment history and stable employment.",
    "Low-risk borrower seeking funds for planned expenses.",
    "Financially responsible individual with consistent income.",
    "Applicant demonstrates strong creditworthiness and stable finances."
]

safe_closers = [
    "No prior defaults. Excellent credit standing.",
    "Applicant poses minimal credit risk.",
    "All financial obligations are currently met.",
    "Borrower has maintained a clean financial record.",
    "Risk assessment indicates low probability of default."
]

rng = np.random.default_rng(42)

def build_adversarial_text(row):
    return (
        f"{safe_openers[rng.integers(len(safe_openers))]} "
        f"Purpose: {safe_purposes[rng.integers(len(safe_purposes))]}. "
        f"Title: {safe_titles[rng.integers(len(safe_titles))]}. "
        f"Employment: {safe_emp_titles[rng.integers(len(safe_emp_titles))]}. "
        f"Annual income: ${int(row['annual_inc'])}. "
        f"Debt-to-income ratio: {row['dti']:.2f}. "
        f"Loan amount: ${int(row['loan_amnt'])}. "
        f"Loan-to-income ratio: {row['loan_to_income']:.3f}. "
        f"{safe_closers[rng.integers(len(safe_closers))]}"
    )

# Overwrite the original text with adversarially phrased text
adv_df["text"] = adv_df.apply(build_adversarial_text, axis=1)

print("\nSample adversarial text:")
print(adv_df["text"].iloc[0])

# Scale numeric columns the same way as train/val/test
adv_df[NUM_COLS] = scaler.transform(adv_df[NUM_COLS])

adv_ds = Dataset.from_pandas(
    adv_df[["text", "label", "reg_violation"] + NUM_COLS + flag_cols],
    preserve_index=False
).map(
    lambda batch: tokenizer(batch["text"], truncation=True, max_length=96),
    batched=True
)

adv_ds = adv_ds.remove_columns(["text"])

adv_df.to_csv(ADV_PATH, index=False)
print(f"\nAdversarial dataset ready: {len(adv_ds)} examples — saved to {ADV_PATH}")

### Side-by-side compliance comparison (clean + adversarial)

In [ ]:
def compliance_metrics(trainer_obj, eval_ds, raw_df, label, threshold=0.5):
    logits = trainer_obj.predict(eval_ds).predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()

    df = raw_df.reset_index(drop=True).copy()
    df["p_low_risk"] = probs[:, 0]
    df["p_risky"] = probs[:, 1]
    df["pred"] = (df["p_risky"] >= threshold).astype(int)

    viol = df[df["reg_violation"] == 1]
    viol_risky = viol[viol["label"] == 1]

    return {
        "Mean P(low risk) on violating cases":
            viol["p_low_risk"].mean() if len(viol) > 0 else np.nan,
        "False low-risk rate on violating risky cases":
            (viol_risky["pred"] == 0).mean() if len(viol_risky) > 0 else np.nan,
        "Recall for risky class on violating cases":
            (viol_risky["pred"] == 1).mean() if len(viol_risky) > 0 else np.nan,
    }

# --- Clean test set comparison
b_clean = compliance_metrics(baseline_trainer, test_ds, test_df, "Baseline", threshold=TAU)
r_clean = compliance_metrics(reg_trainer, test_ds, test_df, "Regulatory", threshold=TAU)

print("\n── Clean test set ──")
print(
    pd.DataFrame(
        [b_clean, r_clean],
        index=["Baseline (λ=0.0)", f"Regulatory (λ={LAMBDA_REG})"]
    ).to_string(float_format=lambda x: f"{x:.4f}")
)

# --- Adversarial test set comparison
b_adv = compliance_metrics(baseline_trainer, adv_ds, adv_df, "Baseline", threshold=TAU)
r_adv = compliance_metrics(reg_trainer, adv_ds, adv_df, "Regulatory", threshold=TAU)

print("\n── Adversarial test set ──")
print(
    pd.DataFrame(
        [b_adv, r_adv],
        index=["Baseline (λ=0.0)", f"Regulatory (λ={LAMBDA_REG})"]
    ).to_string(float_format=lambda x: f"{x:.4f}")
)

## FURTHER METRICS

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset
from scipy.stats import spearmanr
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)
from sklearn.calibration import calibration_curve

sns.set_theme(style="whitegrid")
EVAL_THRESHOLD = TAU

def build_eval_dataset(df):
    ds = Dataset.from_pandas(
        df[["text", "label", "reg_violation"] + NUM_COLS + flag_cols],
        preserve_index=False
    ).map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=96),
        batched=True
    )
    ds = ds.remove_columns(["text"])
    return ds

def get_probs_preds(trainer_obj, eval_ds, threshold=EVAL_THRESHOLD):
    pred_output = trainer_obj.predict(eval_ds)
    logits = pred_output.predictions
    if isinstance(logits, tuple):
        logits = logits[0]

    probs = torch.softmax(torch.tensor(logits), dim=1).cpu().numpy()
    p_low_risk = probs[:, 0]
    p_risky = probs[:, 1]
    pred = (p_risky >= threshold).astype(int)
    return logits, probs, p_low_risk, p_risky, pred

def binary_metrics_from_probs(y_true, p_risky, threshold=EVAL_THRESHOLD):
    y_true = np.asarray(y_true)
    y_pred = (np.asarray(p_risky) >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

    out = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "False Positive Rate": fpr,
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Brier Score": brier_score_loss(y_true, p_risky),
        "Log Loss": log_loss(y_true, np.column_stack([1 - p_risky, p_risky]), labels=[0, 1]),
    }

    if len(np.unique(y_true)) == 2:
        out["ROC-AUC"] = roc_auc_score(y_true, p_risky)
        out["PR-AUC"] = average_precision_score(y_true, p_risky)
    else:
        out["ROC-AUC"] = np.nan
        out["PR-AUC"] = np.nan

    return out

# Raw, unscaled full test set for compliance/slice analysis
test_df_raw_eval = prepare_df(pd.read_csv(TEST_PATH))

# Raw, clean counterpart of the adversarial subset
clean_adv_df_raw_eval = test_df_raw_eval[
    (test_df_raw_eval["reg_violation"] == 1) &
    (test_df_raw_eval["label"] == 1)
].reset_index(drop=True).copy()

# Scaled version for model inference
clean_adv_df_eval = clean_adv_df_raw_eval.copy()
clean_adv_df_eval[NUM_COLS] = scaler.transform(clean_adv_df_eval[NUM_COLS])

clean_adv_ds = build_eval_dataset(clean_adv_df_eval)

assert len(clean_adv_df_raw_eval) == len(adv_ds), "Paired clean/adversarial sets do not match in length."

# Get predictions once and reuse everywhere
_, _, baseline_test_p_low, baseline_test_p_risky, baseline_test_pred = get_probs_preds(baseline_trainer, test_ds)
_, _, reg_test_p_low, reg_test_p_risky, reg_test_pred = get_probs_preds(reg_trainer, test_ds)

_, _, baseline_clean_adv_p_low, baseline_clean_adv_p_risky, baseline_clean_adv_pred = get_probs_preds(baseline_trainer, clean_adv_ds)
_, _, reg_clean_adv_p_low, reg_clean_adv_p_risky, reg_clean_adv_pred = get_probs_preds(reg_trainer, clean_adv_ds)

_, _, baseline_adv_p_low, baseline_adv_p_risky, baseline_adv_pred = get_probs_preds(baseline_trainer, adv_ds)
_, _, reg_adv_p_low, reg_adv_p_risky, reg_adv_pred = get_probs_preds(reg_trainer, adv_ds)

print("Setup complete.")
print("Clean full test size:", len(test_df_raw_eval))
print("Paired clean/adversarial subset size:", len(clean_adv_df_raw_eval))

In [ ]:
# =========================================================
# 1) CLEAN TEST PERFORMANCE TABLE
# =========================================================
clean_perf_table = pd.DataFrame(
    [
        binary_metrics_from_probs(test_df_raw_eval["label"].values, baseline_test_p_risky, threshold=EVAL_THRESHOLD),
        binary_metrics_from_probs(test_df_raw_eval["label"].values, reg_test_p_risky, threshold=EVAL_THRESHOLD),
    ],
    index=["Baseline (λ=0.0)", f"Regulatory (λ={LAMBDA_REG})"]
)

print("\n── Clean test: main performance table ──")
display(clean_perf_table.style.format("{:.4f}"))


# =========================================================
# 2) COMPLIANCE / REGULATORY TABLE
# =========================================================
def build_compliance_table(raw_df, baseline_p_risky, reg_p_risky, threshold=EVAL_THRESHOLD):
    out_rows = []

    for model_name, p_risky in [
        ("Baseline (λ=0.0)", baseline_p_risky),
        (f"Regulatory (λ={LAMBDA_REG})", reg_p_risky),
    ]:
        df = raw_df.reset_index(drop=True).copy()
        df["p_risky"] = p_risky
        df["p_low_risk"] = 1 - df["p_risky"]
        df["pred"] = (df["p_risky"] >= threshold).astype(int)

        viol = df[df["reg_violation"] == 1]
        nonviol = df[df["reg_violation"] == 0]
        viol_risky = viol[viol["label"] == 1]

        rho, _ = spearmanr(df["violation_score"], df["p_risky"])

        out_rows.append({
            "Model": model_name,
            "Mean p(risky) | violation=0": nonviol["p_risky"].mean() if len(nonviol) else np.nan,
            "Mean p(risky) | violation=1": viol["p_risky"].mean() if len(viol) else np.nan,
            "Inconsistency Rate | violation=1": (viol["pred"] == 0).mean() if len(viol) else np.nan,
            "Inconsistency Rate | violation=1,label=1": (viol_risky["pred"] == 0).mean() if len(viol_risky) else np.nan,
            "Threshold Breach Rate (p_low_risk > TAU)": (viol["p_low_risk"] > threshold).mean() if len(viol) else np.nan,
            "Spearman(violation_score, p_risky)": rho,
        })

    return pd.DataFrame(out_rows).set_index("Model")

compliance_table = build_compliance_table(
    test_df_raw_eval,
    baseline_test_p_risky,
    reg_test_p_risky,
    threshold=EVAL_THRESHOLD
)

print("\n── Clean test: compliance table ──")
display(compliance_table.style.format("{:.4f}"))


# =========================================================
# 3) ADVERSARIAL ROBUSTNESS TABLE
# =========================================================
def attack_success_rate(clean_pred, adv_pred):
    clean_pred = np.asarray(clean_pred)
    adv_pred = np.asarray(adv_pred)
    denom = (clean_pred == 1).sum()
    if denom == 0:
        return np.nan
    return ((clean_pred == 1) & (adv_pred == 0)).sum() / denom

robustness_table = pd.DataFrame(
    [
        {
            "Clean Recall (paired subset)": recall_score(clean_adv_df_raw_eval["label"].values, baseline_clean_adv_pred, zero_division=0),
            "Adversarial Recall": recall_score(adv_df["label"].values, baseline_adv_pred, zero_division=0),
            "Recall Drop": recall_score(clean_adv_df_raw_eval["label"].values, baseline_clean_adv_pred, zero_division=0)
                           - recall_score(adv_df["label"].values, baseline_adv_pred, zero_division=0),
            "Attack Success Rate": attack_success_rate(baseline_clean_adv_pred, baseline_adv_pred),
            "Mean Δp(risky) [adv-clean]": (baseline_adv_p_risky - baseline_clean_adv_p_risky).mean(),
            "Mean P(low risk) on adversarial cases": baseline_adv_p_low.mean(),
            "False low-risk rate on adversarial risky cases": (baseline_adv_pred == 0).mean(),
        },
        {
            "Clean Recall (paired subset)": recall_score(clean_adv_df_raw_eval["label"].values, reg_clean_adv_pred, zero_division=0),
            "Adversarial Recall": recall_score(adv_df["label"].values, reg_adv_pred, zero_division=0),
            "Recall Drop": recall_score(clean_adv_df_raw_eval["label"].values, reg_clean_adv_pred, zero_division=0)
                           - recall_score(adv_df["label"].values, reg_adv_pred, zero_division=0),
            "Attack Success Rate": attack_success_rate(reg_clean_adv_pred, reg_adv_pred),
            "Mean Δp(risky) [adv-clean]": (reg_adv_p_risky - reg_clean_adv_p_risky).mean(),
            "Mean P(low risk) on adversarial cases": reg_adv_p_low.mean(),
            "False low-risk rate on adversarial risky cases": (reg_adv_pred == 0).mean(),
        },
    ],
    index=["Baseline (λ=0.0)", f"Regulatory (λ={LAMBDA_REG})"]
)

print("\n── Adversarial robustness table ──")
display(robustness_table.style.format("{:.4f}"))


# =========================================================
# 4) VIOLATION-SCORE SLICE TABLE
# =========================================================
slice_df = test_df_raw_eval.reset_index(drop=True).copy()
slice_df["baseline_p_risky"] = baseline_test_p_risky
slice_df["baseline_pred"] = baseline_test_pred
slice_df["reg_p_risky"] = reg_test_p_risky
slice_df["reg_pred"] = reg_test_pred

violation_score_table = (
    slice_df
    .groupby("violation_score")
    .apply(
        lambda g: pd.Series({
            "N": len(g),
            "Label Rate": g["label"].mean(),
            "Baseline Mean p(risky)": g["baseline_p_risky"].mean(),
            "Regulatory Mean p(risky)": g["reg_p_risky"].mean(),
            "Baseline Recall (risky only)": recall_score(g.loc[g["label"] == 1, "label"], g.loc[g["label"] == 1, "baseline_pred"], zero_division=0) if (g["label"] == 1).any() else np.nan,
            "Regulatory Recall (risky only)": recall_score(g.loc[g["label"] == 1, "label"], g.loc[g["label"] == 1, "reg_pred"], zero_division=0) if (g["label"] == 1).any() else np.nan,
            "Baseline False low-risk rate (risky only)": (g.loc[g["label"] == 1, "baseline_pred"] == 0).mean() if (g["label"] == 1).any() else np.nan,
            "Regulatory False low-risk rate (risky only)": (g.loc[g["label"] == 1, "reg_pred"] == 0).mean() if (g["label"] == 1).any() else np.nan,
        })
    )
    .reset_index()
    .sort_values("violation_score")
)

print("\n── Violation-score slice table ──")
display(violation_score_table.style.format("{:.4f}"))


# Optional: save tables
clean_perf_table.to_csv("clean_performance_table.csv")
compliance_table.to_csv("compliance_table.csv")
robustness_table.to_csv("adversarial_robustness_table.csv")
violation_score_table.to_csv("violation_score_table.csv", index=False)

print("Saved:")
print("- clean_performance_table.csv")
print("- compliance_table.csv")
print("- adversarial_robustness_table.csv")
print("- violation_score_table.csv")

In [ ]:
y_test = test_df_raw_eval["label"].values

# ROC
fpr_b, tpr_b, _ = roc_curve(y_test, baseline_test_p_risky)
fpr_r, tpr_r, _ = roc_curve(y_test, reg_test_p_risky)

roc_auc_b = roc_auc_score(y_test, baseline_test_p_risky)
roc_auc_r = roc_auc_score(y_test, reg_test_p_risky)

# PR
prec_b, rec_b, _ = precision_recall_curve(y_test, baseline_test_p_risky)
prec_r, rec_r, _ = precision_recall_curve(y_test, reg_test_p_risky)

pr_auc_b = average_precision_score(y_test, baseline_test_p_risky)
pr_auc_r = average_precision_score(y_test, reg_test_p_risky)

# Calibration
prob_true_b, prob_pred_b = calibration_curve(y_test, baseline_test_p_risky, n_bins=10, strategy="quantile")
prob_true_r, prob_pred_r = calibration_curve(y_test, reg_test_p_risky, n_bins=10, strategy="quantile")

brier_b = brier_score_loss(y_test, baseline_test_p_risky)
brier_r = brier_score_loss(y_test, reg_test_p_risky)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# ROC
axes[0].plot(fpr_b, tpr_b, label=f"Baseline (AUC={roc_auc_b:.4f})", linewidth=2)
axes[0].plot(fpr_r, tpr_r, label=f"Regulatory (AUC={roc_auc_r:.4f})", linewidth=2)
axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC Curve — Clean Test")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

# PR
axes[1].plot(rec_b, prec_b, label=f"Baseline (PR-AUC={pr_auc_b:.4f})", linewidth=2)
axes[1].plot(rec_r, prec_r, label=f"Regulatory (PR-AUC={pr_auc_r:.4f})", linewidth=2)
axes[1].set_title("Precision-Recall Curve — Clean Test")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

# Calibration
axes[2].plot(prob_pred_b, prob_true_b, marker="o", label=f"Baseline (Brier={brier_b:.4f})", linewidth=2)
axes[2].plot(prob_pred_r, prob_true_r, marker="o", label=f"Regulatory (Brier={brier_r:.4f})", linewidth=2)
axes[2].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[2].set_title("Calibration Curve — Clean Test")
axes[2].set_xlabel("Mean Predicted Probability")
axes[2].set_ylabel("Observed Positive Rate")
axes[2].legend()

plt.tight_layout()
plt.savefig("ROC_PR_CALIB.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# 1) VIOLATION SCORE MONOTONICITY PLOT
# =========================================================
plot_df = test_df_raw_eval.reset_index(drop=True).copy()
plot_df["baseline_p_risky"] = baseline_test_p_risky
plot_df["reg_p_risky"] = reg_test_p_risky

score_plot = (
    plot_df
    .groupby("violation_score")
    .agg(
        N=("label", "size"),
        Baseline=("baseline_p_risky", "mean"),
        Regulatory=("reg_p_risky", "mean"),
    )
    .reset_index()
)

plt.figure(figsize=(8, 5))
plt.plot(score_plot["violation_score"], score_plot["Baseline"], marker="o", linewidth=2, label="Baseline")
plt.plot(score_plot["violation_score"], score_plot["Regulatory"], marker="o", linewidth=2, label="Regulatory")
plt.title("Mean Predicted Risky Probability vs Violation Score")
plt.xlabel("Violation Score")
plt.ylabel("Mean p(risky)")
plt.legend()
plt.tight_layout()
plt.show()


# =========================================================
# 2) THRESHOLD SWEEP ON ADVERSARIAL SET
# =========================================================
thresholds = np.linspace(0.05, 0.95, 19)

threshold_rows = []
for thr in thresholds:
    b_pred = (baseline_adv_p_risky >= thr).astype(int)
    r_pred = (reg_adv_p_risky >= thr).astype(int)

    threshold_rows.append({
        "threshold": thr,
        "Baseline False low-risk rate": (b_pred == 0).mean(),
        "Regulatory False low-risk rate": (r_pred == 0).mean(),
        "Baseline Recall": recall_score(adv_df["label"].values, b_pred, zero_division=0),
        "Regulatory Recall": recall_score(adv_df["label"].values, r_pred, zero_division=0),
    })

threshold_sweep_table = pd.DataFrame(threshold_rows)
threshold_sweep_table.to_csv("threshold_sweep_table.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    threshold_sweep_table["threshold"],
    threshold_sweep_table["Baseline False low-risk rate"],
    marker="o", linewidth=2, label="Baseline"
)
axes[0].plot(
    threshold_sweep_table["threshold"],
    threshold_sweep_table["Regulatory False low-risk rate"],
    marker="o", linewidth=2, label="Regulatory"
)
axes[0].axvline(EVAL_THRESHOLD, linestyle="--", color="gray", label=f"TAU={EVAL_THRESHOLD:.2f}")
axes[0].set_title("Adversarial Safety Curve")
axes[0].set_xlabel("Decision Threshold")
axes[0].set_ylabel("False Low-Risk Rate")
axes[0].legend()

axes[1].plot(
    threshold_sweep_table["threshold"],
    threshold_sweep_table["Baseline Recall"],
    marker="o", linewidth=2, label="Baseline"
)
axes[1].plot(
    threshold_sweep_table["threshold"],
    threshold_sweep_table["Regulatory Recall"],
    marker="o", linewidth=2, label="Regulatory"
)
axes[1].axvline(EVAL_THRESHOLD, linestyle="--", color="gray", label=f"TAU={EVAL_THRESHOLD:.2f}")
axes[1].set_title("Adversarial Recall vs Threshold")
axes[1].set_xlabel("Decision Threshold")
axes[1].set_ylabel("Recall")
axes[1].legend()

plt.tight_layout()
plt.savefig("violation_score_plot.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: threshold_sweep_table.csv")

In [ ]:
paired_shift_df = pd.DataFrame({
    "baseline_clean_p_risky": baseline_clean_adv_p_risky,
    "baseline_adv_p_risky": baseline_adv_p_risky,
    "reg_clean_p_risky": reg_clean_adv_p_risky,
    "reg_adv_p_risky": reg_adv_p_risky,
})

paired_shift_df["baseline_delta_p_risky"] = paired_shift_df["baseline_adv_p_risky"] - paired_shift_df["baseline_clean_p_risky"]
paired_shift_df["reg_delta_p_risky"] = paired_shift_df["reg_adv_p_risky"] - paired_shift_df["reg_clean_p_risky"]

paired_shift_df.to_csv("paired_probability_shifts.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of delta p(risky)
sns.histplot(
    paired_shift_df["baseline_delta_p_risky"],
    bins=30, kde=True, color="tab:blue", alpha=0.45, label="Baseline", ax=axes[0]
)
sns.histplot(
    paired_shift_df["reg_delta_p_risky"],
    bins=30, kde=True, color="tab:orange", alpha=0.45, label="Regulatory", ax=axes[0]
)
axes[0].axvline(0, linestyle="--", color="black")
axes[0].set_title("Distribution of Probability Shift: adv - clean")
axes[0].set_xlabel("Δp(risky)")
axes[0].legend()

# Scatter clean vs adversarial
axes[1].scatter(
    paired_shift_df["baseline_clean_p_risky"],
    paired_shift_df["baseline_adv_p_risky"],
    alpha=0.35, s=18, label="Baseline"
)
axes[1].scatter(
    paired_shift_df["reg_clean_p_risky"],
    paired_shift_df["reg_adv_p_risky"],
    alpha=0.35, s=18, label="Regulatory"
)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="black")
axes[1].set_title("Clean vs Adversarial p(risky)")
axes[1].set_xlabel("Clean p(risky)")
axes[1].set_ylabel("Adversarial p(risky)")
axes[1].legend()

plt.tight_layout()
plt.savefig("Paired_clean_vs_adversarial_probability_shift_plots.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: paired_probability_shifts.csv")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_b = confusion_matrix(test_df_raw_eval["label"].values, baseline_test_pred, labels=[0, 1])
cm_r = confusion_matrix(test_df_raw_eval["label"].values, reg_test_pred, labels=[0, 1])

sns.heatmap(cm_b, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("Baseline Confusion Matrix — Clean Test")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")
axes[0].set_xticklabels(["Low Risk", "Risky"])
axes[0].set_yticklabels(["Low Risk", "Risky"], rotation=0)

sns.heatmap(cm_r, annot=True, fmt="d", cmap="Oranges", cbar=False, ax=axes[1])
axes[1].set_title("Regulatory Confusion Matrix — Clean Test")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")
axes[1].set_xticklabels(["Low Risk", "Risky"])
axes[1].set_yticklabels(["Low Risk", "Risky"], rotation=0)

plt.tight_layout()
plt.savefig("Confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()

# The following gives concrete examples of:
(OPTIONAL)

**cases where baseline is fooled but regulatory is not**

**cases where regulatory is fooled but baseline is not**


In [ ]:
qual_df = clean_adv_df_raw_eval.copy()
qual_df["clean_text"] = clean_adv_df_raw_eval["text"].values
qual_df["adversarial_text"] = adv_df["text"].values

qual_df["baseline_clean_pred"] = baseline_clean_adv_pred
qual_df["baseline_adv_pred"] = baseline_adv_pred
qual_df["reg_clean_pred"] = reg_clean_adv_pred
qual_df["reg_adv_pred"] = reg_adv_pred

qual_df["baseline_clean_p_risky"] = baseline_clean_adv_p_risky
qual_df["baseline_adv_p_risky"] = baseline_adv_p_risky
qual_df["reg_clean_p_risky"] = reg_clean_adv_p_risky
qual_df["reg_adv_p_risky"] = reg_adv_p_risky

# Baseline fooled, regulatory resists
baseline_fooled_reg_ok = qual_df[
    (qual_df["baseline_adv_pred"] == 0) &
    (qual_df["reg_adv_pred"] == 1)
].copy()

# Regulatory fooled, baseline resists
reg_fooled_baseline_ok = qual_df[
    (qual_df["baseline_adv_pred"] == 1) &
    (qual_df["reg_adv_pred"] == 0)
].copy()

print("\n── Baseline fooled, Regulatory correct ──")
display(
    baseline_fooled_reg_ok[
        [
            "violation_score",
            "annual_inc", "dti", "loan_amnt",
            "baseline_adv_p_risky", "reg_adv_p_risky",
            "adversarial_text"
        ]
    ].head(10)
)

print("\n── Regulatory fooled, Baseline correct ──")
display(
    reg_fooled_baseline_ok[
        [
            "violation_score",
            "annual_inc", "dti", "loan_amnt",
            "baseline_adv_p_risky", "reg_adv_p_risky",
            "adversarial_text"
        ]
    ].head(10)
)